# Summation and Product Notation — Exercises

20 graded exercises covering every section of [notes.md](notes.md) and [theory.ipynb](theory.ipynb).

**Format**: Each exercise has:
- 🎯 **Problem** — real-world ML context
- 📝 **Scaffold** — fill in `# YOUR CODE HERE`
- ✅ **Solution** — in a separate cell, with assertions and explanations

| Level | Description | Exercises |
|-------|-------------|----------|
| ⭐ | Essential — core summation/product concepts | 1–6 |
| ⭐⭐ | Applied — real ML summation patterns | 7–14 |
| ⭐⭐⭐ | Interview/Production — advanced patterns & debugging | 15–20 |

| Section Covered | Exercises |
|----------------|----------|
| Basic Summation & Product Manipulation | 1, 2 |
| Algebraic Properties (Linearity, Telescoping) | 3, 4 |
| Summation Formulas (Arithmetic, Geometric) | 5, 6 |
| Softmax & Partition Functions | 7, 8 |
| Cross-Entropy & Loss Functions | 9, 10 |
| Einstein Summation & einsum | 11, 12 |
| Double Sums & Index Manipulation | 13, 14 |
| Convergence of Series | 15, 16 |
| Probability & Entropy | 17, 18 |
| Perplexity, BM25 & Production Patterns | 19, 20 |

In [ ]:
import numpy as np
from fractions import Fraction
import math

np.random.seed(42)
np.set_printoptions(precision=8, suppress=True)

try:
    import torch
    import torch.nn.functional as F
    torch.manual_seed(42)
    HAS_TORCH = True
    print(f'NumPy {np.__version__} | PyTorch {torch.__version__} — Ready!')
except ImportError:
    HAS_TORCH = False
    print(f'NumPy {np.__version__} — Ready! (PyTorch exercises will use NumPy fallbacks)')

---

## Exercise 1: Expand and Compute Summations ⭐

Before using compact Σ notation fluently, you must be able to **expand** it term by term.
This is the single most important skill for reading ML papers.

**Task**: Implement functions that:
1. Expand $\sum_{k=1}^{n} f(k)$ into a list of terms, then sum them
2. Expand $\prod_{k=1}^{n} f(k)$ into a list of factors, then multiply them
3. Verify against closed-form formulas

**Requirements**:
- No use of built-in `sum()` or `math.prod()` — implement the accumulation loop yourself
- Handle the empty case ($n < $ start) correctly: empty sum = 0, empty product = 1
- Print each term as you accumulate it

In [ ]:
# Exercise 1: Your Solution

def expand_sum(f, start: int, end: int) -> float:
    """Compute Σ_{k=start}^{end} f(k) by explicit accumulation.
    
    Print each term and running total.
    Return the final sum.
    Empty sum (end < start) should return 0.
    """
    # YOUR CODE HERE
    pass

def expand_product(f, start: int, end: int) -> float:
    """Compute Π_{k=start}^{end} f(k) by explicit accumulation.
    
    Print each factor and running product.
    Return the final product.
    Empty product (end < start) should return 1.
    """
    # YOUR CODE HERE
    pass

# Test 1: Σ_{k=1}^{5} (2k - 1) = 1 + 3 + 5 + 7 + 9 = 25
print('=== Sum of first 5 odd numbers ===')
result = expand_sum(lambda k: 2*k - 1, 1, 5)
print(f'Result: {result}, Expected: 25\n')

# Test 2: Π_{j=2}^{5} (j/(j-1)) = 2/1 · 3/2 · 4/3 · 5/4 = 5
print('=== Telescoping product ===')
result = expand_product(lambda j: j / (j - 1), 2, 5)
print(f'Result: {result}, Expected: 5.0\n')

# Test 3: Empty cases
print('=== Empty cases ===')
print(f'Empty sum: {expand_sum(lambda k: k, 5, 3)}, Expected: 0')
print(f'Empty product: {expand_product(lambda k: k, 5, 3)}, Expected: 1')

In [ ]:
# Exercise 1: Solution ✅

def expand_sum(f, start: int, end: int) -> float:
    """Compute Σ_{k=start}^{end} f(k) by explicit accumulation."""
    total = 0  # Identity element for addition
    if end < start:
        print('  (empty sum → 0)')
        return total
    for k in range(start, end + 1):
        term = f(k)
        total += term
        print(f'  k={k}: f({k}) = {term}, running total = {total}')
    return total

def expand_product(f, start: int, end: int) -> float:
    """Compute Π_{k=start}^{end} f(k) by explicit accumulation."""
    product = 1  # Identity element for multiplication
    if end < start:
        print('  (empty product → 1)')
        return product
    for k in range(start, end + 1):
        factor = f(k)
        product *= factor
        print(f'  k={k}: f({k}) = {factor}, running product = {product}')
    return product

# Test 1: Σ_{k=1}^{5} (2k - 1) = 25  (sum of first n odd numbers = n²)
print('=== Sum of first 5 odd numbers ===')
result = expand_sum(lambda k: 2*k - 1, 1, 5)
assert result == 25, f'Expected 25, got {result}'
print(f'✓ Result: {result} = 5² (sum of first n odd numbers = n²)\n')

# Test 2: Π_{j=2}^{5} (j/(j-1)) = 5  (telescoping product)
print('=== Telescoping product ===')
result = expand_product(lambda j: j / (j - 1), 2, 5)
assert abs(result - 5.0) < 1e-10, f'Expected 5.0, got {result}'
print(f'✓ Result: {result} (telescope: 2/1 · 3/2 · 4/3 · 5/4 = 5/1)\n')

# Test 3: Empty cases — identity elements
print('=== Empty cases ===')
empty_s = expand_sum(lambda k: k, 5, 3)
empty_p = expand_product(lambda k: k, 5, 3)
assert empty_s == 0, f'Empty sum should be 0, got {empty_s}'
assert empty_p == 1, f'Empty product should be 1, got {empty_p}'
print(f'✓ Empty sum = {empty_s}, Empty product = {empty_p}')
print('\nKey insight: 0 is the identity for +, 1 is the identity for ×')

---

## Exercise 2: Write Sigma and Pi Notation ⭐

The reverse skill: given an explicit sum, express it in compact notation and verify computationally.

**Task**: For each expression below, identify:
1. The index variable and its range
2. The general term $f(k)$
3. Implement and verify

**Expressions**:
- (a) $3 + 6 + 9 + 12 + \cdots + 3n$
- (b) $1 \cdot 2 + 2 \cdot 3 + 3 \cdot 4 + \cdots + n(n+1)$
- (c) $\frac{1}{1 \cdot 2} + \frac{1}{2 \cdot 3} + \frac{1}{3 \cdot 4} + \cdots + \frac{1}{n(n+1)}$
- (d) $1! + 2! + 3! + \cdots + n!$ (sum of factorials)

In [ ]:
# Exercise 2: Your Solution

n = 10  # Test with n = 10

# (a) 3 + 6 + 9 + ... + 3n
# Sigma notation: Σ_{k=?}^{?} ???
# YOUR CODE HERE
result_a = 0  # Replace with your implementation

# (b) 1·2 + 2·3 + 3·4 + ... + n(n+1)
# Sigma notation: Σ_{k=?}^{?} ???
# YOUR CODE HERE
result_b = 0  # Replace with your implementation

# (c) 1/(1·2) + 1/(2·3) + ... + 1/(n(n+1))
# Sigma notation: Σ_{k=?}^{?} ???
# Hint: This telescopes! What is the closed form?
# YOUR CODE HERE
result_c = 0  # Replace with your implementation

# (d) 1! + 2! + 3! + ... + n!
# Sigma notation: Σ_{k=?}^{?} ???
# YOUR CODE HERE
result_d = 0  # Replace with your implementation

print(f'(a) Sum of 3k for k=1..{n}: {result_a}')
print(f'(b) Sum of k(k+1) for k=1..{n}: {result_b}')
print(f'(c) Sum of 1/(k(k+1)) for k=1..{n}: {result_c}')
print(f'(d) Sum of k! for k=1..{n}: {result_d}')

In [ ]:
# Exercise 2: Solution ✅

n = 10

# (a) 3 + 6 + 9 + ... + 3n = Σ_{k=1}^{n} 3k = 3·Σ_{k=1}^{n} k = 3·n(n+1)/2
result_a = 0
for k in range(1, n + 1):
    result_a += 3 * k
closed_form_a = 3 * n * (n + 1) // 2
assert result_a == closed_form_a
print(f'(a) Σ_{{k=1}}^{{{n}}} 3k = {result_a}')
print(f'    Closed form: 3·n(n+1)/2 = 3·{n}·{n+1}/2 = {closed_form_a} ✓')
print(f'    Property used: linearity — factor out constant 3\n')

# (b) Σ_{k=1}^{n} k(k+1) = Σ k² + Σ k = n(n+1)(2n+1)/6 + n(n+1)/2 = n(n+1)(n+2)/3
result_b = 0
for k in range(1, n + 1):
    result_b += k * (k + 1)
closed_form_b = n * (n + 1) * (n + 2) // 3
assert result_b == closed_form_b
print(f'(b) Σ_{{k=1}}^{{{n}}} k(k+1) = {result_b}')
print(f'    Closed form: n(n+1)(n+2)/3 = {n}·{n+1}·{n+2}/3 = {closed_form_b} ✓')
print(f'    Property used: split k(k+1) = k² + k, apply sum of squares + sum of integers\n')

# (c) Σ_{k=1}^{n} 1/(k(k+1)) — telescopes via partial fractions:
#     1/(k(k+1)) = 1/k - 1/(k+1)
#     Sum = (1 - 1/2) + (1/2 - 1/3) + ... + (1/n - 1/(n+1)) = 1 - 1/(n+1) = n/(n+1)
result_c = 0
for k in range(1, n + 1):
    result_c += 1 / (k * (k + 1))
closed_form_c = n / (n + 1)
assert abs(result_c - closed_form_c) < 1e-12
print(f'(c) Σ_{{k=1}}^{{{n}}} 1/(k(k+1)) = {result_c:.10f}')
print(f'    Closed form: n/(n+1) = {n}/{n+1} = {closed_form_c:.10f} ✓')
print(f'    Property used: partial fractions → telescoping\n')

# (d) Σ_{k=1}^{n} k! — no known closed form, but we can compute it
result_d = 0
for k in range(1, n + 1):
    result_d += math.factorial(k)
print(f'(d) Σ_{{k=1}}^{{{n}}} k! = {result_d}')
print(f'    No closed form exists for this sum.')
print(f'    Note: n! dominates — for large n, Σ k! ≈ n! · (1 + 1/n + ...)')
print(f'    Ratio test: last term {math.factorial(n)} is {math.factorial(n)/result_d:.1%} of total')

---

## Exercise 3: Linearity and Telescoping ⭐

Two of the most used algebraic properties of summation:
- **Linearity**: $\sum (\alpha a_i + \beta b_i) = \alpha \sum a_i + \beta \sum b_i$
- **Telescoping**: $\sum_{i=1}^{n} (a_i - a_{i-1}) = a_n - a_0$

**Task**:
1. Verify linearity numerically: show that splitting the sum gives the same result
2. Implement a general telescoping sum and verify it collapses
3. Use telescoping to prove: $\sum_{i=1}^{n} \frac{1}{i(i+1)} = 1 - \frac{1}{n+1}$
4. **AI Application**: Show that the residual connection $y_L = x_0 + \sum_{l=1}^{L} F_l(y_{l-1})$ is a telescoping pattern

In [ ]:
# Exercise 3: Your Solution

# Part 1: Linearity verification
# Given: a = [3, 1, 4, 1, 5, 9, 2, 6], b = [2, 7, 1, 8, 2, 8, 1, 8]
# Show: Σ (2aᵢ + 3bᵢ) = 2·Σaᵢ + 3·Σbᵢ

a = np.array([3, 1, 4, 1, 5, 9, 2, 6], dtype=float)
b = np.array([2, 7, 1, 8, 2, 8, 1, 8], dtype=float)
alpha, beta = 2.0, 3.0

# Method 1: Sum the combined expression
# YOUR CODE HERE
combined = 0

# Method 2: Separate sums, then combine
# YOUR CODE HERE
separate = 0

print(f'Combined: {combined}, Separate: {separate}, Equal: {combined == separate}')

# Part 2: General telescoping
# Implement: given a sequence [a_0, a_1, ..., a_n], compute Σ(a_i - a_{i-1})
# and verify it equals a_n - a_0

def telescoping_sum(seq):
    """Compute Σ_{i=1}^{n} (a_i - a_{i-1}) the long way, and verify."""
    # YOUR CODE HERE
    pass

test_seq = [10, 25, 17, 42, 33, 88]
telescoping_sum(test_seq)

# Part 3: Telescoping proof of Σ 1/(k(k+1)) = 1 - 1/(n+1)
# Use partial fractions: 1/(k(k+1)) = 1/k - 1/(k+1)
# YOUR CODE HERE

# Part 4: ResNet residual as telescoping
# y_0 = x (input)
# y_l = y_{l-1} + F_l(y_{l-1})  for l = 1, ..., L
# Show y_L = x + Σ_{l=1}^{L} F_l(y_{l-1})
# YOUR CODE HERE

In [ ]:
# Exercise 3: Solution ✅

# Part 1: Linearity verification
a = np.array([3, 1, 4, 1, 5, 9, 2, 6], dtype=float)
b = np.array([2, 7, 1, 8, 2, 8, 1, 8], dtype=float)
alpha, beta = 2.0, 3.0

# Method 1: Sum the combined expression Σ (αaᵢ + βbᵢ)
combined = 0
for i in range(len(a)):
    combined += alpha * a[i] + beta * b[i]

# Method 2: α·Σaᵢ + β·Σbᵢ
sum_a = 0
for x in a:
    sum_a += x
sum_b = 0
for x in b:
    sum_b += x
separate = alpha * sum_a + beta * sum_b

assert combined == separate, f'{combined} ≠ {separate}'
print(f'Part 1: Linearity of summation')
print(f'  Σ(2aᵢ + 3bᵢ) = {combined}')
print(f'  2·Σaᵢ + 3·Σbᵢ = {alpha}·{sum_a} + {beta}·{sum_b} = {separate}')
print(f'  Equal: ✓\n')

# Part 2: General telescoping
def telescoping_sum(seq):
    n = len(seq) - 1
    total = 0
    print(f'  Sequence: {seq}')
    print(f'  Terms: ', end='')
    terms = []
    for i in range(1, len(seq)):
        diff = seq[i] - seq[i-1]
        total += diff
        terms.append(f'({seq[i]}-{seq[i-1]})')
    print(' + '.join(terms))
    shortcut = seq[-1] - seq[0]
    assert total == shortcut, f'{total} ≠ {shortcut}'
    print(f'  Long way: {total}')
    print(f'  Shortcut (a_n - a_0): {seq[-1]} - {seq[0]} = {shortcut} ✓')

print('Part 2: Telescoping sum')
telescoping_sum([10, 25, 17, 42, 33, 88])
print()

# Part 3: Telescoping proof
print('Part 3: Σ 1/(k(k+1)) via telescoping')
n = 20
total = 0
print('  Partial fractions: 1/(k(k+1)) = 1/k - 1/(k+1)')
for k in range(1, n + 1):
    # Each term is (1/k - 1/(k+1)) — the 1/(k+1) cancels with next 1/k
    total += 1/k - 1/(k+1)
closed = 1 - 1/(n+1)
assert abs(total - closed) < 1e-12
print(f'  Σ_{{k=1}}^{{{n}}} 1/(k(k+1)) = {total:.10f}')
print(f'  1 - 1/{n+1} = {closed:.10f} ✓')
print(f'  Cancellation: (1-1/2)+(1/2-1/3)+...+(1/{n}-1/{n+1}) = 1-1/{n+1}\n')

# Part 4: ResNet residual as telescoping
print('Part 4: ResNet residual connection as telescoping')
L = 4  # number of layers
d = 3  # dimension
x = np.array([1.0, 2.0, 3.0])  # input

# Simulate residual blocks: F_l is a simple transformation
np.random.seed(0)
y = x.copy()  # y_0 = x
F_outputs = []
for l in range(1, L + 1):
    F_l = 0.1 * np.random.randn(d)  # small perturbation
    F_outputs.append(F_l)
    y = y + F_l  # y_l = y_{l-1} + F_l(y_{l-1})

# Verify: y_L = x + Σ F_l
y_direct = x.copy()
for F_l in F_outputs:
    y_direct += F_l

assert np.allclose(y, y_direct)
print(f'  Input x = {x}')
print(f'  After {L} residual blocks: y_L = {y}')
print(f'  Direct: x + Σ F_l = {y_direct}')
print(f'  Equal: ✓')
print(f'  This is why ResNets work: output = input + sum of learned residuals')
print(f'  Gradient flows directly through the + (identity skip connection)')

---

## Exercise 4: Index Shifting and Interchange of Order ⭐

Two powerful manipulation techniques:
- **Index shifting**: $\sum_{i=1}^{n} a_i = \sum_{j=0}^{n-1} a_{j+1}$ (substitution $j = i - 1$)
- **Interchange of order**: $\sum_{i=1}^{m} \sum_{j=1}^{n} a_{ij} = \sum_{j=1}^{n} \sum_{i=1}^{m} a_{ij}$

**Task**:
1. Verify index shifting numerically for several substitutions
2. Show that summing a matrix row-first vs column-first gives the same result
3. Change order of summation for a triangular region: $\sum_{i=1}^{n} \sum_{j=i}^{n}$ → $\sum_{j=1}^{n} \sum_{i=1}^{j}$
4. Demonstrate Gauss's trick as a special case of index reversal

In [ ]:
# Exercise 4: Your Solution

n = 8

# Part 1: Index shifting
# Show Σ_{i=1}^{n} i² = Σ_{j=0}^{n-1} (j+1)²
# YOUR CODE HERE
original = 0
shifted = 0

print(f'Original Σ_{{i=1}}^{{{n}}} i² = {original}')
print(f'Shifted Σ_{{j=0}}^{{{n-1}}} (j+1)² = {shifted}')

# Part 2: Row-first vs column-first summation of a matrix
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

# Row-first: Σ_i Σ_j A[i,j]
# YOUR CODE HERE
row_first = 0

# Column-first: Σ_j Σ_i A[i,j]
# YOUR CODE HERE
col_first = 0

print(f'\nRow-first: {row_first}, Column-first: {col_first}')

# Part 3: Triangular region order change
# Compute Σ_{i=1}^{4} Σ_{j=i}^{4} (i+j) both ways
# YOUR CODE HERE

# Part 4: Gauss's trick — sum i from 1 to n using forward + reverse
# S = Σ_{i=1}^{n} i
# S = Σ_{i=1}^{n} (n + 1 - i)   (reverse)
# 2S = Σ_{i=1}^{n} (n + 1) = n(n+1)
# YOUR CODE HERE

In [ ]:
# Exercise 4: Solution ✅

n = 8

# Part 1: Index shifting — substitution j = i - 1
original = 0
for i in range(1, n + 1):
    original += i * i

shifted = 0
for j in range(0, n):
    shifted += (j + 1) ** 2

assert original == shifted
print(f'Part 1: Index shifting')
print(f'  Σ_{{i=1}}^{{{n}}} i² = {original}')
print(f'  Σ_{{j=0}}^{{{n-1}}} (j+1)² = {shifted} ✓')
print(f'  Substitution: j = i-1, so i = j+1, and when i=1→j=0, i={n}→j={n-1}\n')

# Part 2: Row-first vs column-first
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

row_first = 0
for i in range(3):
    for j in range(3):
        row_first += A[i, j]

col_first = 0
for j in range(3):
    for i in range(3):
        col_first += A[i, j]

assert row_first == col_first
print(f'Part 2: Interchange of summation order')
print(f'  Matrix A = \n{A}')
print(f'  Row-first:  Σ_i Σ_j A[i,j] = {row_first}')
print(f'  Col-first:  Σ_j Σ_i A[i,j] = {col_first} ✓')
print(f'  For finite sums over rectangular regions, order always commutes.\n')

# Part 3: Triangular region
print(f'Part 3: Triangular region order change')
m = 4

# Original: Σ_{i=1}^{4} Σ_{j=i}^{4} (i+j)
way1 = 0
pairs1 = []
for i in range(1, m + 1):
    for j in range(i, m + 1):
        way1 += i + j
        pairs1.append((i, j))

# Swapped: Σ_{j=1}^{4} Σ_{i=1}^{j} (i+j)
way2 = 0
pairs2 = []
for j in range(1, m + 1):
    for i in range(1, j + 1):
        way2 += i + j
        pairs2.append((i, j))

assert way1 == way2
assert sorted(pairs1) == sorted(pairs2)
print(f'  Σ_{{i=1}}^{{{m}}} Σ_{{j=i}}^{{{m}}} (i+j) = {way1}')
print(f'  Σ_{{j=1}}^{{{m}}} Σ_{{i=1}}^{{j}} (i+j) = {way2} ✓')
print(f'  Same {len(pairs1)} (i,j) pairs visited: {sorted(pairs1)}\n')

# Part 4: Gauss's trick
print(f'Part 4: Gauss\'s trick via index reversal')
n = 100
S_forward = 0
S_reverse = 0
for i in range(1, n + 1):
    S_forward += i
    S_reverse += (n + 1 - i)

assert S_forward == S_reverse  # Both are S
assert S_forward == n * (n + 1) // 2
print(f'  S (forward) = Σ_{{i=1}}^{{{n}}} i = {S_forward}')
print(f'  S (reverse) = Σ_{{i=1}}^{{{n}}} ({n+1}-i) = {S_reverse}')
print(f'  2S = Σ_{{i=1}}^{{{n}}} ({n+1}) = {n}·{n+1} = {n*(n+1)}')
print(f'  S = {n*(n+1)//2} ✓')
print(f'  This is how 10-year-old Gauss computed 1+2+...+100 = 5050')

---

## Exercise 5: Geometric Series — Finite and Infinite ⭐

The geometric series is the most important formula in applied mathematics and appears everywhere in AI:
- Discount factors in RL: $\sum_{t=0}^{\infty} \gamma^t r_t$
- EMA in Adam optimiser: $\beta_1^t$ decay
- ALiBi positional bias: geometric decay of attention

**Task**:
1. Derive the finite geometric sum $\sum_{i=0}^{n-1} r^i = \frac{1 - r^n}{1 - r}$ numerically
2. Show convergence of the infinite geometric series for $|r| < 1$
3. Compute the derivative series: $\sum_{k=1}^{n} k r^{k-1} = \frac{1 - (n+1)r^n + n r^{n+1}}{(1-r)^2}$
4. **AI Application**: Compute the effective window of an EMA with $\beta = 0.999$

In [ ]:
# Exercise 5: Your Solution

# Part 1: Finite geometric series
# Verify Σ_{i=0}^{n-1} r^i = (1 - r^n) / (1 - r) for various r and n
# YOUR CODE HERE

# Part 2: Infinite geometric series convergence
# Plot partial sums S_N = Σ_{i=0}^{N} r^i and show they approach 1/(1-r)
# YOUR CODE HERE

# Part 3: Derivative series Σ_{k=1}^{n} k·r^{k-1}
# Hint: differentiate both sides of Σ r^k = (1-r^{n+1})/(1-r) with respect to r
# YOUR CODE HERE

# Part 4: EMA effective window
# In Adam, the first moment estimate is: m_t = β·m_{t-1} + (1-β)·g_t
# Unrolling: m_t = (1-β) Σ_{k=0}^{t-1} β^k · g_{t-k}
# What fraction of m_t comes from the last W steps?
# Fraction = (1-β) Σ_{k=0}^{W-1} β^k = 1 - β^W
# Find W such that fraction ≥ 0.95 for β = 0.999
# YOUR CODE HERE

In [ ]:
# Exercise 5: Solution ✅

# Part 1: Finite geometric series
print('Part 1: Finite geometric series Σ_{i=0}^{n-1} r^i = (1 - r^n)/(1 - r)')
for r in [0.5, 2.0, 0.9, 1.5]:
    for n in [5, 10, 20]:
        # Compute by explicit accumulation
        S = 0
        for i in range(n):
            S += r ** i
        # Closed form
        closed = (1 - r**n) / (1 - r)
        assert abs(S - closed) < 1e-8, f'Mismatch for r={r}, n={n}'
    print(f'  r = {r}: all n values match ✓')
print()

# Part 2: Infinite geometric series convergence
print('Part 2: Convergence of infinite geometric series')
for r in [0.5, 0.9, 0.99, 0.999]:
    limit = 1 / (1 - r)
    # Find N such that S_N is within 0.1% of limit
    S = 0
    for N in range(10000):
        S += r ** N
        if abs(S - limit) / limit < 0.001:
            break
    print(f'  r = {r}: 1/(1-r) = {limit:.4f}, S_{{{N}}} = {S:.4f} (converged in {N+1} terms)')

print(f'\n  Rule of thumb: need ~1/(1-r) terms to converge')
print(f'  For r=0.999: need ~1000 terms (≈ 1/(1-0.999))\n')

# Part 3: Derivative series
print('Part 3: Derivative trick — Σ k·r^{k-1}')
r = 0.8
n = 15
# Direct computation
S_direct = 0
for k in range(1, n + 1):
    S_direct += k * r ** (k - 1)

# Closed form from differentiating geometric series
# d/dr [Σ_{k=0}^{n} r^k] = Σ_{k=1}^{n} k·r^{k-1} = d/dr [(1-r^{n+1})/(1-r)]
# = [-(n+1)r^n(1-r) + (1-r^{n+1})] / (1-r)^2
# = [1 - (n+1)r^n + n·r^{n+1}] / (1-r)^2
S_closed = (1 - (n+1) * r**n + n * r**(n+1)) / (1 - r)**2

assert abs(S_direct - S_closed) < 1e-8
print(f'  r = {r}, n = {n}')
print(f'  Direct sum: {S_direct:.8f}')
print(f'  Closed form: {S_closed:.8f} ✓')
print(f'  This derivative trick extends: Σ k²r^k, Σ k³r^k, etc.\n')

# Part 4: EMA effective window
print('Part 4: EMA effective window in Adam optimiser')
beta = 0.999
# Fraction of weight in last W steps = 1 - β^W
# Solve 1 - β^W ≥ 0.95 → β^W ≤ 0.05 → W ≥ log(0.05)/log(β)
target_fractions = [0.50, 0.90, 0.95, 0.99]
print(f'  β = {beta}')
print(f'  EMA: m_t = (1-β) Σ_{{k=0}}^{{t-1}} β^k · g_{{t-k}}')
print(f'  Weight in last W steps = 1 - β^W\n')
for frac in target_fractions:
    W = math.ceil(math.log(1 - frac) / math.log(beta))
    # Verify
    weight = 0
    for k in range(W):
        weight += (1 - beta) * beta ** k
    assert abs(weight - (1 - beta**W)) < 1e-10
    print(f'  {frac:.0%} of weight in last W = {W} steps (1-β^{W} = {1-beta**W:.4f})')

print(f'\n  Key insight: β=0.999 means ~3000 step effective window.')
print(f'  This is why Adam "remembers" gradients from thousands of steps ago.')

---

## Exercise 6: Product-to-Sum via Logarithm ⭐

The single most important conversion in ML: $\log \prod_i a_i = \sum_i \log a_i$.

This transforms multiplying many small probabilities (which underflows) into adding log-probabilities (which is stable).

**Task**:
1. Compute a product of probabilities directly and observe underflow
2. Compute the same product in log-space and convert back
3. Implement negative log-likelihood (NLL) loss from scratch
4. Show that maximising $\prod p(x_i | \theta)$ ≡ minimising $-\sum \log p(x_i | \theta)$
5. Demonstrate factorial via the product-to-sum identity: $\ln(n!) = \sum_{k=1}^{n} \ln k$

In [ ]:
# Exercise 6: Your Solution

# Part 1: Probability product underflow
# Simulate 500 i.i.d. word probabilities, each around 0.01
np.random.seed(42)
probs = np.random.uniform(0.005, 0.02, size=500)

# Direct product
# YOUR CODE HERE
direct_product = 1.0

print(f'Direct product of {len(probs)} probabilities: {direct_product}')
print(f'(Expect 0.0 due to underflow)')

# Part 2: Log-space computation
# YOUR CODE HERE
log_sum = 0.0
log_product = 0.0  # exp(log_sum)

print(f'Log-space sum: {log_sum}')
print(f'Product via exp(log_sum): {log_product}')

# Part 3: NLL loss
# Given token probabilities, compute NLL = -Σ log p(x_i)
# YOUR CODE HERE

# Part 4: Show max likelihood ≡ min NLL
# YOUR CODE HERE

# Part 5: Stirling check — ln(n!) = Σ ln(k)
# YOUR CODE HERE

In [ ]:
# Exercise 6: Solution ✅

# Part 1: Direct product underflow
np.random.seed(42)
probs = np.random.uniform(0.005, 0.02, size=500)

direct_product = 1.0
for p in probs:
    direct_product *= p
print(f'Part 1: Direct product of {len(probs)} small probabilities')
print(f'  Product = {direct_product}')
print(f'  Underflowed to zero: {direct_product == 0.0}')
print(f'  Smallest float64 ≈ {np.finfo(np.float64).tiny:.2e}\n')

# Part 2: Log-space computation
log_sum = 0.0
for p in probs:
    log_sum += np.log(p)
print(f'Part 2: Log-space computation')
print(f'  Σ log(pᵢ) = {log_sum:.4f}')
print(f'  This is finite! log-space avoids underflow.')
print(f'  Average log-prob per token: {log_sum / len(probs):.4f}')
print(f'  (Would need ~{-log_sum / np.log(10):.0f} digits to represent product directly)\n')

# Part 3: NLL loss
# Simulate a language model: 20 tokens, each with a predicted probability
np.random.seed(123)
token_probs = np.random.uniform(0.01, 0.3, size=20)
nll = 0.0
for p in token_probs:
    nll += -np.log(p)
avg_nll = nll / len(token_probs)
print(f'Part 3: Negative Log-Likelihood')
print(f'  Token probabilities: {token_probs[:5]}... (first 5 of {len(token_probs)})')
print(f'  NLL = -Σ log p(xᵢ) = {nll:.4f}')
print(f'  Average NLL per token = {avg_nll:.4f}')
print(f'  Perplexity = exp(avg NLL) = {np.exp(avg_nll):.2f}\n')

# Part 4: Max likelihood ≡ Min NLL
print(f'Part 4: max Π p(xᵢ) ≡ min -Σ log p(xᵢ)')
# Consider two models with different probability assignments
probs_model_A = np.array([0.1, 0.2, 0.3, 0.15, 0.05])
probs_model_B = np.array([0.2, 0.3, 0.1, 0.25, 0.15])

# Likelihood = Π p(xi)
lik_A = 1.0
lik_B = 1.0
for p in probs_model_A:
    lik_A *= p
for p in probs_model_B:
    lik_B *= p

# NLL = -Σ log p(xi)
nll_A = 0.0
nll_B = 0.0
for p in probs_model_A:
    nll_A += -np.log(p)
for p in probs_model_B:
    nll_B += -np.log(p)

better_by_lik = 'A' if lik_A > lik_B else 'B'
better_by_nll = 'A' if nll_A < nll_B else 'B'
assert better_by_lik == better_by_nll, 'Rankings should agree!'

print(f'  Model A: Likelihood = {lik_A:.6f}, NLL = {nll_A:.4f}')
print(f'  Model B: Likelihood = {lik_B:.6f}, NLL = {nll_B:.4f}')
print(f'  Better by likelihood: Model {better_by_lik}')
print(f'  Better by NLL (lower): Model {better_by_nll} ✓')
print(f'  Rankings agree: max likelihood ≡ min NLL\n')

# Part 5: Stirling's approximation via sum of logs
print(f'Part 5: ln(n!) = Σ ln(k) and Stirling\'s approximation')
for n in [5, 10, 50, 100]:
    # Exact: Σ_{k=1}^{n} ln(k)
    log_factorial_exact = 0.0
    for k in range(1, n + 1):
        log_factorial_exact += np.log(k)

    # Stirling: n·ln(n) - n + 0.5·ln(2πn)
    stirling = n * np.log(n) - n + 0.5 * np.log(2 * np.pi * n)

    rel_error = abs(stirling - log_factorial_exact) / abs(log_factorial_exact) * 100
    print(f'  n={n:>3}: ln(n!) = {log_factorial_exact:.4f}, Stirling = {stirling:.4f}, error = {rel_error:.3f}%')

print(f'\n  Stirling improves as 1/(12n). Essential for large-scale combinatorics in AI.')

---

## Exercise 7: Softmax from Scratch ⭐⭐

Softmax is THE summation pattern in modern AI: $\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$

It converts raw logits into a probability distribution. The denominator (partition function) is a single summation that normalises everything.

**Task**:
1. Implement naive softmax (will overflow for large logits)
2. Implement numerically stable softmax using the log-sum-exp trick
3. Verify that softmax outputs sum to 1
4. Show that softmax is invariant to constant shifts: $\text{softmax}(z + c) = \text{softmax}(z)$
5. Compute the Jacobian $\frac{\partial \text{softmax}(z)_i}{\partial z_j}$ numerically

In [ ]:
# Exercise 7: Your Solution

def softmax_naive(z):
    """Naive softmax — will overflow for large logits."""
    # YOUR CODE HERE
    # softmax(z)_i = exp(z_i) / Σ_j exp(z_j)
    pass

def softmax_stable(z):
    """Numerically stable softmax using max subtraction."""
    # YOUR CODE HERE
    # Hint: subtract max(z) from all elements first
    pass

# Test with normal logits
z_normal = np.array([2.0, 1.0, 0.5, -1.0])
print('Normal logits:', z_normal)
print('Naive:', softmax_naive(z_normal))
print('Stable:', softmax_stable(z_normal))

# Test with large logits (should overflow naively)
z_large = np.array([1000.0, 999.0, 998.0, 997.0])
print('\nLarge logits:', z_large)
print('Naive:', softmax_naive(z_large))
print('Stable:', softmax_stable(z_large))

# Verify sum to 1
# YOUR CODE HERE

# Verify shift invariance
# YOUR CODE HERE

# Compute Jacobian (bonus)
# ∂softmax(z)_i / ∂z_j = softmax(z)_i · (δ_{ij} - softmax(z)_j)
# YOUR CODE HERE

In [ ]:
# Exercise 7: Solution ✅

import warnings

def softmax_naive(z):
    """Naive softmax — overflows for large logits."""
    z = np.array(z, dtype=float)
    exp_z = np.zeros_like(z)
    for i in range(len(z)):
        exp_z[i] = np.exp(z[i])
    partition = 0.0
    for val in exp_z:
        partition += val
    result = np.zeros_like(z)
    for i in range(len(z)):
        result[i] = exp_z[i] / partition
    return result

def softmax_stable(z):
    """Numerically stable softmax: subtract max before exp."""
    z = np.array(z, dtype=float)
    # Find max
    m = z[0]
    for val in z:
        if val > m:
            m = val
    # Compute exp(z - max)
    exp_z = np.zeros_like(z)
    for i in range(len(z)):
        exp_z[i] = np.exp(z[i] - m)
    # Partition function
    partition = 0.0
    for val in exp_z:
        partition += val
    # Normalise
    result = np.zeros_like(z)
    for i in range(len(z)):
        result[i] = exp_z[i] / partition
    return result

# Test 1: Normal logits
z_normal = np.array([2.0, 1.0, 0.5, -1.0])
naive_result = softmax_naive(z_normal)
stable_result = softmax_stable(z_normal)
print('Part 1: Normal logits')
print(f'  z = {z_normal}')
print(f'  Naive:  {naive_result}')
print(f'  Stable: {stable_result}')
assert np.allclose(naive_result, stable_result)
print(f'  Match: ✓\n')

# Test 2: Large logits (naive overflows)
z_large = np.array([1000.0, 999.0, 998.0, 997.0])
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    naive_result = softmax_naive(z_large)
stable_result = softmax_stable(z_large)
print('Part 2: Large logits (overflow test)')
print(f'  z = {z_large}')
print(f'  Naive:  {naive_result} (nan due to inf/inf)')
print(f'  Stable: {stable_result} ✓\n')

# Part 3: Sum to 1
print('Part 3: Probabilities sum to 1')
for z in [z_normal, np.random.randn(10), np.array([-5, 0, 5, 10])]:
    s = softmax_stable(z)
    total = 0.0
    for p in s:
        total += p
    assert abs(total - 1.0) < 1e-10
print(f'  All test cases: Σ softmax(z)_i = 1.0 ✓\n')

# Part 4: Shift invariance
print('Part 4: Shift invariance — softmax(z + c) = softmax(z)')
z = np.array([1.0, 3.0, -2.0, 0.5])
for c in [-100, 0, 42, 1000]:
    z_shifted = z + c
    s_original = softmax_stable(z)
    s_shifted = softmax_stable(z_shifted)
    assert np.allclose(s_original, s_shifted)
print(f'  All shifts c ∈ {{-100, 0, 42, 1000}}: identical output ✓')
print(f'  Proof: exp(z_i + c)/Σ exp(z_j + c) = e^c·exp(z_i)/(e^c·Σ exp(z_j)) = exp(z_i)/Σ exp(z_j)\n')

# Part 5: Jacobian
print('Part 5: Softmax Jacobian')
z = np.array([2.0, 1.0, 0.5, -1.0])
s = softmax_stable(z)
K = len(z)

# Analytical: ∂s_i/∂z_j = s_i(δ_{ij} - s_j)
J_analytical = np.zeros((K, K))
for i in range(K):
    for j in range(K):
        if i == j:
            J_analytical[i, j] = s[i] * (1 - s[j])
        else:
            J_analytical[i, j] = -s[i] * s[j]

# Numerical: finite differences
eps = 1e-7
J_numerical = np.zeros((K, K))
for j in range(K):
    z_plus = z.copy()
    z_plus[j] += eps
    z_minus = z.copy()
    z_minus[j] -= eps
    J_numerical[:, j] = (softmax_stable(z_plus) - softmax_stable(z_minus)) / (2 * eps)

assert np.allclose(J_analytical, J_numerical, atol=1e-5)
print(f'  s = {s}')
print(f'  Jacobian (analytical):\n{np.array2string(J_analytical, precision=4)}')
print(f'  Matches numerical Jacobian: ✓')
print(f'  Key property: each row sums to 0 (probabilities are constrained to sum to 1)')

---

## Exercise 8: Cross-Entropy Loss Decomposition ⭐⭐

The cross-entropy loss decomposes into summation layers:

$$\mathcal{L} = -\frac{1}{B} \sum_{i=1}^{B} \sum_{c=1}^{C} y_{ic} \log \hat{y}_{ic}$$

For one-hot targets, the inner sum collapses to a single term. Understanding this collapse and the gradient structure is essential.

**Task**:
1. Implement cross-entropy with one-hot targets (full double sum)
2. Show the inner sum collapses: only the correct-class term survives
3. Implement the combined softmax + cross-entropy (numerically stable)
4. Verify the gradient: $\frac{\partial \mathcal{L}}{\partial z_k} = \hat{y}_k - y_k$
5. Implement label smoothing and show its effect

In [ ]:
# Exercise 8: Your Solution

def cross_entropy_full(y_onehot, y_hat):
    """Cross-entropy using full double sum: -Σ_c y_c log(ŷ_c)."""
    # YOUR CODE HERE
    pass

def cross_entropy_collapsed(y_class, y_hat):
    """Cross-entropy using collapsed form: -log(ŷ_{correct_class})."""
    # YOUR CODE HERE
    pass

def softmax_cross_entropy(logits, y_class):
    """Numerically stable combined softmax + cross-entropy.
    
    Loss = -z_c + log(Σ exp(z_j))
    Use log-sum-exp trick for stability.
    """
    # YOUR CODE HERE
    pass

# Test data
logits = np.array([2.0, 1.0, 0.1, -1.0, 0.5])
correct_class = 0  # True class is 0
C = len(logits)

# Test cross-entropy both ways
# YOUR CODE HERE

# Compute gradient ∂L/∂z_k and verify it equals softmax(z)_k - y_k
# YOUR CODE HERE

# Label smoothing: y_smooth = (1-ε)·onehot + ε/C
# YOUR CODE HERE

In [ ]:
# Exercise 8: Solution ✅

def cross_entropy_full(y_onehot, y_hat):
    """Cross-entropy using full double sum: -Σ_c y_c log(ŷ_c)."""
    loss = 0.0
    for c in range(len(y_onehot)):
        if y_onehot[c] > 0:  # Avoid log(0) when y=0
            loss += -y_onehot[c] * np.log(y_hat[c])
    return loss

def cross_entropy_collapsed(y_class, y_hat):
    """Cross-entropy for one-hot: -log(ŷ_{correct_class})."""
    return -np.log(y_hat[y_class])

def softmax_cross_entropy(logits, y_class):
    """Stable softmax + CE: -z_c + log Σ exp(z_j) = -z_c + logsumexp(z)."""
    z = np.array(logits, dtype=float)
    m = z[0]
    for val in z:
        if val > m:
            m = val
    # log-sum-exp: m + log(Σ exp(z_j - m))
    exp_sum = 0.0
    for val in z:
        exp_sum += np.exp(val - m)
    logsumexp = m + np.log(exp_sum)
    return -z[y_class] + logsumexp

# Test data
logits = np.array([2.0, 1.0, 0.1, -1.0, 0.5])
correct_class = 0
C = len(logits)

# Compute softmax probabilities
probs = softmax_stable(logits)

# One-hot encoding
y_onehot = np.zeros(C)
y_onehot[correct_class] = 1.0

# Compare all three methods
loss_full = cross_entropy_full(y_onehot, probs)
loss_collapsed = cross_entropy_collapsed(correct_class, probs)
loss_stable = softmax_cross_entropy(logits, correct_class)

print('Part 1-3: Cross-entropy three ways')
print(f'  Logits: {logits}')
print(f'  Softmax: {probs}')
print(f'  Correct class: {correct_class}')
print(f'  Full double sum:     {loss_full:.6f}')
print(f'  Collapsed (one-hot): {loss_collapsed:.6f}')
print(f'  Stable (log-space):  {loss_stable:.6f}')
assert abs(loss_full - loss_collapsed) < 1e-10
assert abs(loss_full - loss_stable) < 1e-10
print(f'  All three agree: ✓\n')

# Part 4: Gradient verification
print('Part 4: Gradient ∂L/∂z_k = softmax(z)_k - y_k')
# Analytical gradient
grad_analytical = np.zeros(C)
for k in range(C):
    grad_analytical[k] = probs[k] - y_onehot[k]

# Numerical gradient (finite differences)
eps = 1e-7
grad_numerical = np.zeros(C)
for k in range(C):
    logits_plus = logits.copy()
    logits_plus[k] += eps
    logits_minus = logits.copy()
    logits_minus[k] -= eps
    loss_plus = softmax_cross_entropy(logits_plus, correct_class)
    loss_minus = softmax_cross_entropy(logits_minus, correct_class)
    grad_numerical[k] = (loss_plus - loss_minus) / (2 * eps)

assert np.allclose(grad_analytical, grad_numerical, atol=1e-5)
print(f'  Analytical: {grad_analytical}')
print(f'  Numerical:  {grad_numerical}')
print(f'  Match: ✓')
print(f'  Key insight: gradient = (predicted - target) for each class')
print(f'  For correct class: {probs[correct_class]:.4f} - 1 = {probs[correct_class]-1:.4f}')
print(f'  For wrong classes: just the predicted probability (pushes it down)\n')

# Part 5: Label smoothing
print('Part 5: Label smoothing')
epsilon = 0.1
y_smooth = np.zeros(C)
for c in range(C):
    y_smooth[c] = epsilon / C
    if c == correct_class:
        y_smooth[c] += (1 - epsilon)

loss_hard = cross_entropy_full(y_onehot, probs)
loss_smooth = cross_entropy_full(y_smooth, probs)
print(f'  ε = {epsilon}')
print(f'  Hard targets: {y_onehot}')
print(f'  Smooth targets: {y_smooth}')
print(f'  Loss (hard):   {loss_hard:.6f}')
print(f'  Loss (smooth): {loss_smooth:.6f}')
print(f'  Smooth loss is higher — it penalises overconfidence.')
print(f'  Gradient with smoothing pushes non-target logits up slightly → regularisation.')

---

## Exercise 9: Attention as Weighted Sum ⭐⭐

The core attention operation: $\text{Attn}(Q, K, V)_i = \sum_j \alpha_{ij} V_j$ where $\alpha_{ij} = \text{softmax}(QK^T / \sqrt{d_k})_{ij}$

This is a weighted sum of value vectors, where the weights come from softmax of dot-product scores.

**Task**:
1. Implement single-head attention from scratch using only loops (no matrix ops)
2. Count the exact number of multiplications and additions
3. Implement causal (lower-triangular) masking
4. Verify that attention weights sum to 1 for each query position

In [ ]:
# Exercise 9: Your Solution

def attention_manual(Q, K, V, causal=False):
    """Single-head attention using only explicit loops.
    
    Q: (n_q, d_k) — query vectors
    K: (n_k, d_k) — key vectors
    V: (n_k, d_v) — value vectors
    causal: if True, mask future positions (j > i)
    
    Returns: (n_q, d_v) — output vectors, (n_q, n_k) — attention weights
    """
    n_q, d_k = Q.shape
    n_k = K.shape[0]
    d_v = V.shape[1]
    
    # Step 1: Compute scores S[i,j] = Σ_d Q[i,d] · K[j,d] / sqrt(d_k)
    # YOUR CODE HERE
    
    # Step 2: Apply causal mask (set future scores to -inf)
    # YOUR CODE HERE
    
    # Step 3: Apply softmax row-wise to get attention weights
    # YOUR CODE HERE
    
    # Step 4: Compute output O[i,d] = Σ_j α[i,j] · V[j,d]
    # YOUR CODE HERE
    
    pass  # Return (output, attention_weights)

# Test
np.random.seed(42)
n, d_k, d_v = 4, 3, 2
Q = np.random.randn(n, d_k)
K = np.random.randn(n, d_k)
V = np.random.randn(n, d_v)

output, weights = attention_manual(Q, K, V, causal=False)
print('Attention weights:\n', weights)
print('Output:\n', output)

# Verify weights sum to 1 per query
# YOUR CODE HERE

# Count operations
# YOUR CODE HERE

In [ ]:
# Exercise 9: Solution ✅

def attention_manual(Q, K, V, causal=False):
    """Single-head attention using only explicit loops."""
    n_q, d_k = Q.shape
    n_k = K.shape[0]
    d_v = V.shape[1]
    scale = np.sqrt(d_k)
    mult_count = 0
    add_count = 0

    # Step 1: Scores S[i,j] = (Σ_d Q[i,d]·K[j,d]) / sqrt(d_k)
    S = np.zeros((n_q, n_k))
    for i in range(n_q):
        for j in range(n_k):
            dot = 0.0
            for d in range(d_k):
                dot += Q[i, d] * K[j, d]
                mult_count += 1
                add_count += 1
            S[i, j] = dot / scale
            mult_count += 1  # division counts as mult

    # Step 2: Causal mask
    if causal:
        for i in range(n_q):
            for j in range(n_k):
                if j > i:
                    S[i, j] = -1e9  # -inf approximation

    # Step 3: Row-wise softmax
    alpha = np.zeros((n_q, n_k))
    for i in range(n_q):
        # Find max for stability
        row_max = S[i, 0]
        for j in range(1, n_k):
            if S[i, j] > row_max:
                row_max = S[i, j]
        # Exp and sum
        exp_sum = 0.0
        for j in range(n_k):
            alpha[i, j] = np.exp(S[i, j] - row_max)
            add_count += 1
        for j in range(n_k):
            exp_sum += alpha[i, j]
            add_count += 1
        # Normalise
        for j in range(n_k):
            alpha[i, j] /= exp_sum
            mult_count += 1

    # Step 4: Weighted sum O[i,d] = Σ_j α[i,j]·V[j,d]
    O = np.zeros((n_q, d_v))
    for i in range(n_q):
        for d in range(d_v):
            for j in range(n_k):
                O[i, d] += alpha[i, j] * V[j, d]
                mult_count += 1
                add_count += 1

    print(f'  FLOPs: {mult_count} multiplications, {add_count} additions')
    return O, alpha

# Test: bidirectional attention
np.random.seed(42)
n, d_k, d_v = 4, 3, 2
Q = np.random.randn(n, d_k)
K = np.random.randn(n, d_k)
V = np.random.randn(n, d_v)

print('Bidirectional attention:')
output, weights = attention_manual(Q, K, V, causal=False)
print(f'\n  Attention weights (each row sums to 1):')
for i in range(n):
    row_sum = 0.0
    for j in range(n):
        row_sum += weights[i, j]
    print(f'    Query {i}: {weights[i]} → sum = {row_sum:.6f}')
    assert abs(row_sum - 1.0) < 1e-10
print(f'  All rows sum to 1: ✓')

print(f'\n  Output:\n{output}\n')

# Causal attention
print('Causal attention:')
output_causal, weights_causal = attention_manual(Q, K, V, causal=True)
print(f'\n  Causal weights (lower triangular):')
for i in range(n):
    print(f'    Query {i}: {weights_causal[i]}')
    # Check future positions are ~0
    for j in range(i + 1, n):
        assert weights_causal[i, j] < 1e-6, f'Future position {j} not masked for query {i}'
print(f'  Future positions masked: ✓')

# Operation count analysis
print(f'\n  Complexity analysis for n={n}, d_k={d_k}, d_v={d_v}:')
print(f'  QK^T:        n²·d_k = {n}²·{d_k} = {n*n*d_k} mults')
print(f'  Softmax:     n² operations')
print(f'  Attn × V:    n²·d_v = {n}²·{d_v} = {n*n*d_v} mults')
print(f'  Total: O(n²·(d_k + d_v)) = O(n²·d)')
print(f'  This quadratic scaling in n is why long sequences are expensive!')

---

## Exercise 10: Matrix Multiplication as Summation ⭐⭐

Every dense layer in a neural network is matrix multiplication: $C_{ij} = \sum_k A_{ik} B_{kj}$.

**Task**:
1. Implement matrix multiply using only triple nested loops (the raw summation)
2. Count exact FLOPs (multiply-accumulate operations)
3. Implement batched matrix multiply: compute $C_b = A_b B_b$ for $b = 1, \ldots, B$
4. Verify against NumPy's `@` operator
5. Compute the FLOPs for a transformer's forward pass through one layer

In [ ]:
# Exercise 10: Your Solution

def matmul_raw(A, B):
    """Matrix multiply C = A @ B using triple loop.
    
    C[i,j] = Σ_k A[i,k] · B[k,j]
    """
    # YOUR CODE HERE
    pass

def batched_matmul_raw(A, B):
    """Batched matrix multiply: C[b] = A[b] @ B[b] for each b.
    
    A: (batch, m, p)
    B: (batch, p, n)
    Returns: (batch, m, n)
    """
    # YOUR CODE HERE
    pass

# Test single matmul
A = np.array([[1, 2, 3],
              [4, 5, 6]])  # 2x3
B = np.array([[7, 8],
              [9, 10],
              [11, 12]])   # 3x2

C = matmul_raw(A, B)
print('A @ B =\n', C)

# Verify against numpy
# YOUR CODE HERE

# Test batched matmul
# YOUR CODE HERE

# FLOPs for transformer layer
# YOUR CODE HERE

In [ ]:
# Exercise 10: Solution ✅

def matmul_raw(A, B):
    """C[i,j] = Σ_k A[i,k] · B[k,j]"""
    m, p = A.shape
    p2, n = B.shape
    assert p == p2, f'Inner dimensions must match: {p} ≠ {p2}'
    C = np.zeros((m, n))
    flops = 0
    for i in range(m):
        for j in range(n):
            total = 0.0
            for k in range(p):
                total += A[i, k] * B[k, j]
                flops += 2  # one multiply + one add
            C[i, j] = total
    return C, flops

def batched_matmul_raw(A, B):
    """C[b,i,j] = Σ_k A[b,i,k] · B[b,k,j]"""
    batch = A.shape[0]
    m, p = A.shape[1], A.shape[2]
    n = B.shape[2]
    C = np.zeros((batch, m, n))
    total_flops = 0
    for b in range(batch):
        for i in range(m):
            for j in range(n):
                total = 0.0
                for k in range(p):
                    total += A[b, i, k] * B[b, k, j]
                    total_flops += 2
                C[b, i, j] = total
    return C, total_flops

# Test single matmul
A = np.array([[1., 2., 3.],
              [4., 5., 6.]])  # 2x3
B = np.array([[7., 8.],
              [9., 10.],
              [11., 12.]])   # 3x2

C, flops = matmul_raw(A, B)
C_numpy = A @ B
print('Part 1: Raw matrix multiplication')
print(f'  A ({A.shape}) @ B ({B.shape}) = C ({C.shape})')
print(f'  Raw result:\n{C}')
print(f'  NumPy result:\n{C_numpy}')
assert np.allclose(C, C_numpy)
print(f'  Match: ✓')
print(f'  FLOPs: {flops} = 2·m·p·n = 2·{A.shape[0]}·{A.shape[1]}·{B.shape[1]} = {2*A.shape[0]*A.shape[1]*B.shape[1]}\n')

# Test batched matmul
np.random.seed(42)
batch = 4
m, p, n = 3, 5, 2
A_batch = np.random.randn(batch, m, p)
B_batch = np.random.randn(batch, p, n)

C_batch, batch_flops = batched_matmul_raw(A_batch, B_batch)
C_batch_numpy = np.matmul(A_batch, B_batch)
print('Part 2: Batched matrix multiplication')
print(f'  Shape: ({batch}, {m}, {p}) @ ({batch}, {p}, {n}) = ({batch}, {m}, {n})')
assert np.allclose(C_batch, C_batch_numpy)
print(f'  Matches np.matmul: ✓')
print(f'  FLOPs: {batch_flops} = B·2mpn = {batch}·2·{m}·{p}·{n} = {batch*2*m*p*n}\n')

# Part 3: Transformer layer FLOPs
print('Part 3: FLOPs for one transformer layer')
print('  Parameters: d_model=768, n_heads=12, d_ff=3072, seq_len=512\n')
d = 768
n_heads = 12
d_k = d // n_heads  # 64
d_ff = 3072
seq = 512

# QKV projection: 3 × (seq × d) @ (d × d) = 3 × 2·seq·d²
qkv_flops = 3 * 2 * seq * d * d
print(f'  QKV projections:    3 × 2·{seq}·{d}² = {qkv_flops:>15,} FLOPs')

# Attention scores: (seq × d_k) @ (d_k × seq) per head × n_heads = n_heads × 2·seq²·d_k
attn_score_flops = n_heads * 2 * seq * seq * d_k
print(f'  Attention scores:   {n_heads} × 2·{seq}²·{d_k} = {attn_score_flops:>15,} FLOPs')

# Attention × V: (seq × seq) @ (seq × d_k) per head × n_heads
attn_v_flops = n_heads * 2 * seq * seq * d_k
print(f'  Attention × V:      {n_heads} × 2·{seq}²·{d_k} = {attn_v_flops:>15,} FLOPs')

# Output projection: (seq × d) @ (d × d)
out_proj_flops = 2 * seq * d * d
print(f'  Output projection:  2·{seq}·{d}² = {out_proj_flops:>15,} FLOPs')

# FFN: two linear layers (seq × d) @ (d × d_ff) + (seq × d_ff) @ (d_ff × d)
ffn_flops = 2 * (2 * seq * d * d_ff)
print(f'  FFN (2 layers):     2 × 2·{seq}·{d}·{d_ff} = {ffn_flops:>15,} FLOPs')

total = qkv_flops + attn_score_flops + attn_v_flops + out_proj_flops + ffn_flops
print(f'  ────────────────────────────────────────')
print(f'  Total per layer:    {total:>15,} FLOPs')
print(f'  ≈ {total/1e9:.2f} GFLOPs per layer')
print(f'  For 12 layers: ≈ {12*total/1e9:.2f} GFLOPs per forward pass')

---

## Exercise 11: Einstein Summation with einsum ⭐⭐

`np.einsum` and `torch.einsum` translate Einstein notation strings directly into tensor operations. Mastering einsum lets you express complex multi-dimensional operations in one line.

**Task**:
1. Implement these operations using explicit loops, then verify with `np.einsum`:
   - Dot product: `'i,i->'`
   - Matrix multiply: `'ik,kj->ij'`
   - Outer product: `'i,j->ij'`
   - Trace: `'ii->'`
   - Batch matrix multiply: `'bij,bjk->bik'`
2. Implement multi-head attention with einsum strings
3. Explain what each einsum string computes by identifying free vs dummy indices

In [ ]:
# Exercise 11: Your Solution

np.random.seed(42)

# Vectors for dot product and outer product
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

# Matrices for matmul and trace
A = np.random.randn(3, 4)
B = np.random.randn(4, 2)
M = np.random.randn(3, 3)  # square matrix for trace

# Batch matrices
batch_A = np.random.randn(2, 3, 4)
batch_B = np.random.randn(2, 4, 5)

# Part 1: Implement each operation with loops, then verify with einsum

# (a) Dot product: Σ_i a_i · b_i → scalar
# YOUR CODE HERE
dot_loop = 0
dot_einsum = 0

# (b) Matrix multiply: Σ_k A[i,k]·B[k,j] → (3,2)
# YOUR CODE HERE
mm_loop = np.zeros((3, 2))
mm_einsum = np.zeros((3, 2))

# (c) Outer product: a_i · b_j → (3,3)
# YOUR CODE HERE
outer_loop = np.zeros((3, 3))
outer_einsum = np.zeros((3, 3))

# (d) Trace: Σ_i M[i,i] → scalar
# YOUR CODE HERE
trace_loop = 0
trace_einsum = 0

# (e) Batch matmul: Σ_j A[b,i,j]·B[b,j,k] → (2,3,5)
# YOUR CODE HERE
bmm_loop = np.zeros((2, 3, 5))
bmm_einsum = np.zeros((2, 3, 5))

# Part 2: Multi-head attention with einsum
# YOUR CODE HERE

In [ ]:
# Exercise 11: Solution ✅

np.random.seed(42)
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
A = np.random.randn(3, 4)
B = np.random.randn(4, 2)
M = np.random.randn(3, 3)
batch_A = np.random.randn(2, 3, 4)
batch_B = np.random.randn(2, 4, 5)

# (a) Dot product: 'i,i->'
#     Index i appears in both inputs → summed (dummy)
#     No output indices → scalar result
dot_loop = 0.0
for i in range(len(a)):
    dot_loop += a[i] * b[i]
dot_einsum = np.einsum('i,i->', a, b)
assert abs(dot_loop - dot_einsum) < 1e-10
print(f'(a) Dot product: \'i,i->\'')
print(f'    Free: none | Dummy: i')
print(f'    Loop: {dot_loop}, einsum: {dot_einsum} ✓\n')

# (b) Matrix multiply: 'ik,kj->ij'
#     k appears in both inputs → summed (dummy)
#     i,j each appear once in output → free
mm_loop = np.zeros((3, 2))
for i in range(3):
    for j in range(2):
        for k in range(4):
            mm_loop[i, j] += A[i, k] * B[k, j]
mm_einsum = np.einsum('ik,kj->ij', A, B)
assert np.allclose(mm_loop, mm_einsum)
print(f'(b) Matrix multiply: \'ik,kj->ij\'')
print(f'    Free: i,j | Dummy: k')
print(f'    Match: ✓\n')

# (c) Outer product: 'i,j->ij'
#     No index repeated → no summation!
#     Both i,j are free → 2D output
outer_loop = np.zeros((3, 3))
for i in range(3):
    for j in range(3):
        outer_loop[i, j] = a[i] * b[j]
outer_einsum = np.einsum('i,j->ij', a, b)
assert np.allclose(outer_loop, outer_einsum)
print(f'(c) Outer product: \'i,j->ij\'')
print(f'    Free: i,j | Dummy: none (no contraction)')
print(f'    Result:\n{outer_einsum}')
print(f'    Match: ✓\n')

# (d) Trace: 'ii->'
#     i appears twice in same tensor → diagonal elements, then sum
trace_loop = 0.0
for i in range(3):
    trace_loop += M[i, i]
trace_einsum = np.einsum('ii->', M)
assert abs(trace_loop - trace_einsum) < 1e-10
print(f'(d) Trace: \'ii->\'')
print(f'    Free: none | Dummy: i (self-contraction)')
print(f'    Loop: {trace_loop:.4f}, einsum: {trace_einsum:.4f} ✓\n')

# (e) Batch matmul: 'bij,bjk->bik'
#     b appears in both inputs AND output → batch index (not summed)
#     j appears in both inputs but NOT output → summed
#     i,k appear in output → free
bmm_loop = np.zeros((2, 3, 5))
for bb in range(2):
    for i in range(3):
        for k in range(5):
            for j in range(4):
                bmm_loop[bb, i, k] += batch_A[bb, i, j] * batch_B[bb, j, k]
bmm_einsum = np.einsum('bij,bjk->bik', batch_A, batch_B)
assert np.allclose(bmm_loop, bmm_einsum)
print(f'(e) Batch matmul: \'bij,bjk->bik\'')
print(f'    Free: b,i,k | Dummy: j')
print(f'    b is a batch index (kept but not summed)')
print(f'    Match: ✓\n')

# Part 2: Multi-head attention via einsum
print('Part 2: Multi-head attention via einsum')
np.random.seed(0)
B, H, N, D = 2, 4, 6, 8  # batch, heads, seq_len, head_dim

Q = np.random.randn(B, H, N, D)
K = np.random.randn(B, H, N, D)
V = np.random.randn(B, H, N, D)

# Step 1: Scores = Q @ K^T / sqrt(D)
# 'bhqd,bhkd->bhqk' — contract over d (head dim)
scores = np.einsum('bhqd,bhkd->bhqk', Q, K) / np.sqrt(D)
print(f'  Scores shape: {scores.shape} (batch, heads, querylen, keylen)')
print(f'  einsum: \'bhqd,bhkd->bhqk\' — free: b,h,q,k | dummy: d')

# Step 2: Softmax over last axis (key dimension)
# Manual softmax for each (b,h,q) slice
attn = np.zeros_like(scores)
for b_idx in range(B):
    for h_idx in range(H):
        for q_idx in range(N):
            row = scores[b_idx, h_idx, q_idx]
            row_max = row[0]
            for val in row:
                if val > row_max:
                    row_max = val
            exp_row = np.zeros(N)
            exp_sum = 0.0
            for k_idx in range(N):
                exp_row[k_idx] = np.exp(row[k_idx] - row_max)
                exp_sum += exp_row[k_idx]
            for k_idx in range(N):
                attn[b_idx, h_idx, q_idx, k_idx] = exp_row[k_idx] / exp_sum

# Step 3: Output = attn @ V
# 'bhqk,bhkd->bhqd' — contract over k (key positions)
output = np.einsum('bhqk,bhkd->bhqd', attn, V)
print(f'  Attn weights shape: {attn.shape}')
print(f'  Output shape: {output.shape}')
print(f'  einsum: \'bhqk,bhkd->bhqd\' — free: b,h,q,d | dummy: k')
print(f'\n  3 lines of einsum = complete multi-head attention! ✓')

## Exercise 12: Double Sums & Interchange of Order ⭐⭐

**Topics**: Double summation, dependent bounds, order interchange, diagonal extraction

### Tasks

**(a)** Compute $\displaystyle\sum_{i=0}^{3}\sum_{j=0}^{3} (i + j)$ in **both row-major** and **column-major** order using explicit nested loops. Verify they give the same result.

**(b)** Compute the **upper-triangular sum** $\displaystyle\sum_{i=0}^{n-1}\sum_{j=i}^{n-1} A_{ij}$ for a random 5×5 matrix using:
- Nested loops with dependent bound `j = i..n-1`
- Interchanged order: outer `j`, inner `i = 0..j`

Show both match and equal `np.sum(np.triu(A))`.

**(c)** **Diagonal sums**: For a random $n \times n$ matrix, compute the sum along each diagonal $d$ (where $d = j - i$).  
- Diagonal $d$ has entries $(i, i+d)$ for valid $i$.
- Return a dict mapping diagonal index $d$ to its sum.

**(d)** **Convolution as double sum**: Given two 1D arrays $a$ and $b$, compute the convolution
$$c_k = \sum_{j=0}^{k} a_j \cdot b_{k-j}$$
using explicit double loops. Verify against `np.convolve(a, b)`.

In [ ]:
# Exercise 12: Your Solution

import numpy as np

# (a) Double sum: ∑_i ∑_j (i + j) for i,j in 0..3
# Row-major: outer i, inner j
row_major = 0.0
# YOUR CODE HERE

# Column-major: outer j, inner i
col_major = 0.0
# YOUR CODE HERE

print(f'Row-major: {row_major}, Column-major: {col_major}')

# (b) Upper-triangular sum with dependent bounds
n = 5
np.random.seed(12)
A = np.random.randn(n, n)

# Original order: ∑_{i=0}^{n-1} ∑_{j=i}^{n-1} A[i,j]
triu_sum_v1 = 0.0
# YOUR CODE HERE

# Interchanged order: ∑_{j=0}^{n-1} ∑_{i=0}^{j} A[i,j]
triu_sum_v2 = 0.0
# YOUR CODE HERE

print(f'Triu sum (i,j): {triu_sum_v1:.6f}')
print(f'Triu sum (j,i): {triu_sum_v2:.6f}')
print(f'np.triu sum:    {np.sum(np.triu(A)):.6f}')

# (c) Diagonal sums: sum along each diagonal d = j - i
diag_sums = {}
# YOUR CODE HERE

print(f'Diagonal sums: {diag_sums}')

# (d) Convolution as double sum
a_conv = np.array([1.0, 2.0, 3.0])
b_conv = np.array([4.0, 5.0])
# Output length = len(a) + len(b) - 1
out_len = len(a_conv) + len(b_conv) - 1
c_conv = np.zeros(out_len)
# YOUR CODE HERE
# c_k = ∑_{j=0}^{k} a[j] * b[k-j]  (only valid indices)

print(f'Manual conv: {c_conv}')
print(f'np.convolve: {np.convolve(a_conv, b_conv)}')

In [ ]:
# Exercise 12: Solution ✅

import numpy as np

# (a) Double sum: ∑_i ∑_j (i + j), i,j = 0..3
# Row-major order (outer i, inner j)
row_major = 0.0
for i in range(4):
    for j in range(4):
        row_major += (i + j)

# Column-major order (outer j, inner i)
col_major = 0.0
for j in range(4):
    for i in range(4):
        col_major += (i + j)

assert row_major == col_major
print(f'(a) Row-major: {row_major}, Column-major: {col_major} ✓')
print(f'    WHY they match: addition is commutative, order doesn\'t matter')
print(f'    Analytical: 4·∑i + 4·∑j = 4·6 + 4·6 = 48\n')

# (b) Upper-triangular sum with dependent bounds
n = 5
np.random.seed(12)
A = np.random.randn(n, n)

# Original order: ∑_{i=0}^{n-1} ∑_{j=i}^{n-1} A[i,j]
triu_sum_v1 = 0.0
for i in range(n):
    for j in range(i, n):      # j depends on i: j >= i
        triu_sum_v1 += A[i, j]

# Interchanged order: swap loops
# Region: {(i,j) : 0<=i<=j<=n-1}
# Fix j first: j goes 0..n-1, then i goes 0..j
triu_sum_v2 = 0.0
for j in range(n):
    for i in range(j + 1):     # i goes 0..j (since i <= j)
        triu_sum_v2 += A[i, j]

triu_numpy = np.sum(np.triu(A))
assert abs(triu_sum_v1 - triu_sum_v2) < 1e-10
assert abs(triu_sum_v1 - triu_numpy) < 1e-10
print(f'(b) Upper-triangular sum:')
print(f'    Order 1 (i outer): {triu_sum_v1:.6f}')
print(f'    Order 2 (j outer): {triu_sum_v2:.6f}')
print(f'    np.triu check:     {triu_numpy:.6f} ✓')
print(f'    Key insight: when interchanging, transform the bounds!')
print(f'    Original: 0≤i≤n-1, i≤j≤n-1')
print(f'    Swapped:  0≤j≤n-1, 0≤i≤j\n')

# (c) Diagonal sums
diag_sums = {}
for i in range(n):
    for j in range(n):
        d = j - i
        if d not in diag_sums:
            diag_sums[d] = 0.0
        diag_sums[d] += A[i, j]

# Verify against np.trace with offsets
print(f'(c) Diagonal sums (d = j - i):')
for d in sorted(diag_sums.keys()):
    np_val = np.trace(A, offset=d)
    assert abs(diag_sums[d] - np_val) < 1e-10
    print(f'    d={d:+d}: sum={diag_sums[d]:+.4f} (np.trace check ✓)')
print(f'    Main diagonal (d=0) = trace = {diag_sums[0]:.4f}\n')

# (d) Convolution as double sum (Cauchy product)
a_conv = np.array([1.0, 2.0, 3.0])
b_conv = np.array([4.0, 5.0])
out_len = len(a_conv) + len(b_conv) - 1
c_conv = np.zeros(out_len)

for k in range(out_len):
    for j in range(k + 1):
        if j < len(a_conv) and (k - j) < len(b_conv):
            c_conv[k] += a_conv[j] * b_conv[k - j]

c_numpy = np.convolve(a_conv, b_conv)
assert np.allclose(c_conv, c_numpy)
print(f'(d) Convolution (Cauchy product):')
print(f'    c_k = Σ_j a[j]·b[k-j]')
print(f'    a = {a_conv}, b = {b_conv}')
print(f'    Manual:     {c_conv}')
print(f'    np.convolve:{c_numpy} ✓')
print(f'    This is how 1D conv layers work in CNNs!')

## Exercise 13: Kahan Summation & Numerical Stability ⭐⭐

**Topics**: Floating-point catastrophic cancellation, compensated summation, running mean stability

### Tasks

**(a)** Demonstrate **catastrophic cancellation**: sum `1e8` with `1.0` one million times using `float32`. Compare naive loop vs `float64` ground truth. Show how much error accumulates.

**(b)** Implement **Kahan compensated summation** with explicit tracking of the compensation variable `c`. Show it recovers accuracy even in `float32`.

**(c)** Implement **online (running) mean** two ways:
- **Naive**: accumulate sum, divide at end  
- **Welford's**: $\mu_n = \mu_{n-1} + \frac{x_n - \mu_{n-1}}{n}$

Show that Welford's is more numerically stable when values are large (e.g., $10^9 + \epsilon$).

**(d)** Implement **log-sum-exp** from scratch:
$$\log\sum_i e^{x_i} = m + \log\sum_i e^{x_i - m}, \quad m = \max_i x_i$$
Show the naive version overflows for large inputs but the stable version doesn't.

In [ ]:
# Exercise 13: Your Solution

import numpy as np

# (a) Catastrophic cancellation in float32
# Add 1.0 to a running sum starting at 1e8, repeat 1_000_000 times
n_adds = 1_000_000
expected = 1e8 + n_adds * 1.0  # true answer

naive_sum = np.float32(1e8)
# YOUR CODE HERE: loop n_adds times, adding np.float32(1.0)

print(f'Expected: {expected}')
print(f'Naive f32: {naive_sum}')
print(f'Error: {abs(float(naive_sum) - expected)}')

# (b) Kahan compensated summation
kahan_sum = np.float32(1e8)
c = np.float32(0.0)  # compensation variable
# YOUR CODE HERE: implement Kahan summation

print(f'Kahan f32: {kahan_sum}')
print(f'Error: {abs(float(kahan_sum) - expected)}')

# (c) Online mean: Naive vs Welford
values = np.float32(1e9) + np.random.randn(10000).astype(np.float32)

# Naive mean (accumulate then divide)
# YOUR CODE HERE

# Welford's running mean
# YOUR CODE HERE

# (d) Log-sum-exp
x = np.array([1000.0, 1001.0, 1002.0])

# Naive: log(sum(exp(x))) — will overflow!
# YOUR CODE HERE

# Stable: m + log(sum(exp(x - m)))
# YOUR CODE HERE

In [ ]:
# Exercise 13: Solution ✅

import numpy as np

# (a) Catastrophic cancellation in float32
n_adds = 1_000_000
expected = 1e8 + n_adds * 1.0

naive_sum = np.float32(1e8)
for _ in range(n_adds):
    naive_sum = naive_sum + np.float32(1.0)

naive_error = abs(float(naive_sum) - expected)
print(f'(a) Catastrophic cancellation:')
print(f'    Expected:  {expected:.1f}')
print(f'    Naive f32: {float(naive_sum):.1f}')
print(f'    Error:     {naive_error:.1f}')
print(f'    WHY: float32 has ~7 decimal digits of precision')
print(f'         1e8 + 1.0 = 100000001 needs 9 digits → loses the 1.0!\n')

# (b) Kahan compensated summation
kahan_sum = np.float32(1e8)
c = np.float32(0.0)  # running compensation for lost low-order bits

for _ in range(n_adds):
    y = np.float32(1.0) - c        # compensated value to add
    t = kahan_sum + y               # big + small: low bits lost
    c = (t - kahan_sum) - y         # recover what was lost: (big-big) - small
    kahan_sum = t

kahan_error = abs(float(kahan_sum) - expected)
print(f'(b) Kahan compensated summation:')
print(f'    Kahan f32: {float(kahan_sum):.1f}')
print(f'    Error:     {kahan_error:.1f}')
print(f'    Speedup:   {naive_error / max(kahan_error, 1e-30):.0f}x more accurate')
print(f'    HOW: c tracks the rounding error from each addition\n')

# (c) Online mean: Naive vs Welford
np.random.seed(42)
values = np.float32(1e9) + np.random.randn(10000).astype(np.float32)
true_mean = float(np.mean(values.astype(np.float64)))

# Naive: accumulate sum then divide
naive_total = np.float32(0.0)
for v in values:
    naive_total += v
naive_mean = float(naive_total / np.float32(len(values)))

# Welford's online mean: μ_n = μ_{n-1} + (x_n - μ_{n-1}) / n
welford_mean = np.float32(0.0)
for idx, v in enumerate(values):
    n = idx + 1
    welford_mean = welford_mean + (v - welford_mean) / np.float32(n)

print(f'(c) Online mean stability (values ≈ 1e9):')
print(f'    True mean (f64):  {true_mean:.6f}')
print(f'    Naive sum/n:      {naive_mean:.6f}')
print(f'    Welford online:   {float(welford_mean):.6f}')
print(f'    Naive error:      {abs(naive_mean - true_mean):.2f}')
print(f'    Welford error:    {abs(float(welford_mean) - true_mean):.2f}')
print(f'    Welford avoids accumulating a huge sum ≈ n × 1e9\n')

# (d) Log-sum-exp
x = np.array([1000.0, 1001.0, 1002.0])

# Naive: log(sum(exp(x)))
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    exp_x = np.zeros(len(x))
    for i in range(len(x)):
        exp_x[i] = np.exp(x[i])
    naive_lse = np.log(np.sum(exp_x))

print(f'(d) Log-sum-exp:')
print(f'    x = {x}')
print(f'    Naive log(Σexp(x)): {naive_lse}  ← overflow to inf!')

# Stable: m + log(sum(exp(x - m)))
m = x[0]
for val in x:
    if val > m:
        m = val

shifted_exp = np.zeros(len(x))
exp_sum = 0.0
for i in range(len(x)):
    shifted_exp[i] = np.exp(x[i] - m)
    exp_sum += shifted_exp[i]
stable_lse = m + np.log(exp_sum)

print(f'    Stable m + log(Σexp(x-m)): {stable_lse:.6f}  ✓')
print(f'    m = max(x) = {m}')
print(f'    exp(x - m) = {shifted_exp}  ← all ≤ 1, no overflow!')
print(f'    This is used inside every softmax and cross-entropy computation.')

## Exercise 14: Infinite Products & Convergence ⭐⭐

**Topics**: Infinite products, product convergence, Wallis product, product representations

### Tasks

**(a)** Compute the **Wallis product** for π:
$$\frac{\pi}{2} = \prod_{k=1}^{n} \frac{4k^2}{4k^2 - 1}$$
Implement with an explicit loop and track how the partial product converges as $n$ increases.

**(b)** Compute the **Euler product for sin(x)**:
$$\frac{\sin(\pi x)}{\pi x} = \prod_{k=1}^{n}\left(1 - \frac{x^2}{k^2}\right)$$
Verify for $x = 0.5$ that the partial product approaches $\frac{\sin(\pi/2)}{\pi/2} = \frac{2}{\pi}$.

**(c)** **Product convergence test**: An infinite product $\prod(1 + a_k)$ converges absolutely iff $\sum |a_k|$ converges. Verify this by computing:
- $\prod_{k=1}^{n}(1 + 1/k^2)$ → converges (since $\sum 1/k^2 < \infty$)
- $\prod_{k=1}^{n}(1 + 1/k)$ → diverges (since $\sum 1/k = \infty$)

**(d)** **Weight initialization product**: In deep networks, each layer scales activations by a factor. The total scaling after $L$ layers is $\prod_{l=1}^{L} s_l$. 
- If $s_l = 1.01$ for all layers, compute the product for $L = 100, 500, 1000$ (exploding gradients).
- If $s_l = 0.99$, compute for same $L$ values (vanishing gradients).
- Find the $s_l$ that keeps the product ≈ 1 for $L = 100$.

In [ ]:
# Exercise 14: Your Solution

import numpy as np

# (a) Wallis product: π/2 = ∏ 4k²/(4k²-1)
partial_products = []
product = 1.0
for k in range(1, 1001):
    # YOUR CODE HERE
    pass
# Plot or print convergence

# (b) Euler product for sin(πx)/(πx)
x = 0.5
product_euler = 1.0
# YOUR CODE HERE

# (c) Product convergence test
# Convergent: ∏(1 + 1/k²)
prod_converge = 1.0
# YOUR CODE HERE

# Divergent: ∏(1 + 1/k)
prod_diverge = 1.0
# YOUR CODE HERE

# (d) Weight initialization scaling
# YOUR CODE HERE: compute ∏ s_l for s=1.01 and s=0.99

In [ ]:
# Exercise 14: Solution ✅

import numpy as np

# (a) Wallis product: π/2 = ∏_{k=1}^{n} 4k²/(4k²-1)
product = 1.0
milestones = [10, 50, 100, 500, 1000]
print('(a) Wallis product for π/2:')
for k in range(1, 1001):
    numerator = 4.0 * k * k
    denominator = 4.0 * k * k - 1.0
    product *= numerator / denominator
    if k in milestones:
        estimate = 2.0 * product
        error = abs(estimate - np.pi)
        print(f'    n={k:4d}: π ≈ {estimate:.8f}, error = {error:.2e}')
print(f'    True π = {np.pi:.8f}')
print(f'    Converges SLOWLY — O(1/n) rate\n')

# (b) Euler product for sin(πx)/(πx)
x = 0.5
product_euler = 1.0
target = np.sin(np.pi * x) / (np.pi * x)

print(f'(b) Euler product for sin(πx)/(πx) at x={x}:')
for k in range(1, 501):
    product_euler *= (1.0 - x * x / (k * k))
    if k in [10, 50, 100, 500]:
        print(f'    n={k:3d}: product = {product_euler:.8f}')

print(f'    Target sin(π/2)/(π/2) = 2/π = {target:.8f}')
assert abs(product_euler - target) < 1e-4
print(f'    Match ✓\n')

# (c) Product convergence test
print(f'(c) Product convergence:')

# Convergent: ∏(1 + 1/k²)
prod_converge = 1.0
sum_for_test = 0.0
for k in range(1, 10001):
    prod_converge *= (1.0 + 1.0 / (k * k))
    sum_for_test += 1.0 / (k * k)
print(f'    ∏(1+1/k²) for k=1..10000: {prod_converge:.6f}')
print(f'    Σ(1/k²) = {sum_for_test:.6f} < ∞ → product CONVERGES')
# Analytical: sinh(π)/(π) ≈ 3.6762
analytical = np.sinh(np.pi) / np.pi
print(f'    Analytical limit: sinh(π)/π = {analytical:.6f}')

# Divergent: ∏(1 + 1/k)
prod_diverge = 1.0
sum_harmonic = 0.0
for k in range(1, 10001):
    prod_diverge *= (1.0 + 1.0 / k)
    sum_harmonic += 1.0 / k
print(f'    ∏(1+1/k) for k=1..10000: {prod_diverge:.1f}')
print(f'    Σ(1/k) = {sum_harmonic:.4f} → ∞ → product DIVERGES')
print(f'    Note: ∏(1+1/k) = (n+1) exactly (telescope!)\n')

# (d) Weight initialization scaling
print(f'(d) Weight scaling in deep networks:')
for s in [1.01, 0.99]:
    print(f'    s = {s}:')
    for L in [100, 500, 1000]:
        prod = 1.0
        for _ in range(L):
            prod *= s
        print(f'      L={L:4d}: total scale = {prod:.4e}')
    print()

# Find s that keeps product ≈ 1 for L = 100
# s^100 = 1 → s = 1.0
# But in practice, we want s = 1 ± ε
# Xavier init: s = sqrt(2 / (fan_in + fan_out))
# He init:     s = sqrt(2 / fan_in)
print(f'    For s^L ≈ 1, we need s ≈ 1.0 exactly.')
print(f'    Even 1% deviation compounds exponentially!')
print(f'    This is why proper weight initialization matters:')
print(f'    Xavier, He, etc. are designed to make E[s_l] = 1.')

---

# ⭐⭐⭐ Interview & Production Level (Exercises 15-20)

---

## Exercise 15: Series Convergence & Partial Sums ⭐⭐⭐

**Topics**: Convergence tests, partial sums, Taylor series, truncation error analysis

### Tasks

**(a)** Implement the **ratio test** from scratch. Given a series defined by its terms $a_k$, compute
$$L = \lim_{k \to \infty}\left|\frac{a_{k+1}}{a_k}\right|$$
by evaluating the ratio for large $k$. Test on:
- $a_k = \frac{1}{k!}$ → $L = 0$ (converges)
- $a_k = \frac{k^k}{k!}$ → $L = e$ (diverges)
- $a_k = \frac{1}{k^2}$ → $L = 1$ (inconclusive)

**(b)** Compute **Taylor series for $e^x$** using partial sums: $e^x = \sum_{k=0}^{n} \frac{x^k}{k!}$.
- Implement with explicit factorial computation (loop, no `math.factorial`).
- Track the absolute error $|S_n - e^x|$ for each $n$.
- Find the minimum $n$ to reach machine precision for $x = 1, 5, 10$.

**(c)** **Truncation error bound**: For alternating series $\sum (-1)^k a_k$ where $a_k$ decreases to 0, the error after $n$ terms is bounded by $|a_{n+1}|$. Verify this for $\ln(2) = \sum_{k=1}^{\infty} \frac{(-1)^{k+1}}{k}$.

**(d)** **Series acceleration**: Implement **Euler transform** for the alternating harmonic series to speed up convergence to $\ln(2)$. Compare terms needed for 8-digit accuracy.

In [ ]:
# Exercise 15: Your Solution

import numpy as np

# (a) Ratio test: compute |a_{k+1}/a_k| for large k
def ratio_test(term_func, max_k=100):
    """Compute ratio |a_{k+1}/a_k| for increasing k."""
    ratios = []
    for k in range(1, max_k + 1):
        # YOUR CODE HERE: compute ratio, handle division
        pass
    return ratios

# Test series:
# 1/k!, k^k/k!, 1/k²
# YOUR CODE HERE

# (b) Taylor series for e^x with manual factorial
def taylor_exp(x, max_terms=50):
    """Compute e^x via partial sums ∑ x^k / k!"""
    partial_sums = []
    errors = []
    # YOUR CODE HERE: accumulate x^k/k! without using math.factorial
    return partial_sums, errors

# (c) Alternating series error bound for ln(2)
# ln(2) = 1 - 1/2 + 1/3 - 1/4 + ...
# YOUR CODE HERE

# (d) Euler transform acceleration
# YOUR CODE HERE

In [ ]:
# Exercise 15: Solution ✅

import numpy as np

# (a) Ratio test
print('(a) Ratio test: |a_{k+1}/a_k| for large k\n')

# Series 1: a_k = 1/k!
# Compute terms iteratively: a_k = a_{k-1} / k
print('  Series: 1/k!')
prev_term = 1.0  # 1/0! = 1
for k in range(1, 21):
    curr_term = prev_term / k  # 1/k! = (1/(k-1)!) / k
    ratio = curr_term / prev_term
    if k in [1, 5, 10, 20]:
        print(f'    k={k:2d}: ratio = {abs(ratio):.6f}')
    prev_term = curr_term
print(f'    L → 0 < 1: CONVERGES ✓\n')

# Series 2: a_k = k^k / k!
# a_{k+1}/a_k = (k+1)^{k+1}/(k+1)! × k!/k^k = ((k+1)/k)^k
print('  Series: k^k/k!')
for k in [10, 50, 100, 500]:
    ratio = ((k + 1.0) / k) ** k
    print(f'    k={k:3d}: ratio = {ratio:.6f}')
print(f'    L → e ≈ {np.e:.6f} > 1: DIVERGES ✓\n')

# Series 3: a_k = 1/k²
# a_{k+1}/a_k = k²/(k+1)²
print('  Series: 1/k²')
for k in [10, 100, 1000]:
    ratio = (k / (k + 1.0)) ** 2
    print(f'    k={k:4d}: ratio = {ratio:.8f}')
print(f'    L → 1: INCONCLUSIVE (need other tests)\n')

# (b) Taylor series for e^x
print('(b) Taylor series for e^x:')
for x in [1.0, 5.0, 10.0]:
    true_val = np.exp(x)
    partial_sum = 0.0
    term = 1.0  # x^0 / 0! = 1
    min_n = -1
    
    for k in range(60):
        partial_sum += term
        error = abs(partial_sum - true_val)
        if error < 1e-15 and min_n == -1:
            min_n = k
        # Next term: x^{k+1}/(k+1)! = (x^k/k!) × x/(k+1)
        term *= x / (k + 1)
    
    print(f'  x={x:4.1f}: need n={min_n:2d} terms for machine precision')
    print(f'         final error = {abs(partial_sum - true_val):.2e}')

print(f'  Note: larger x needs more terms (slower convergence)\n')

# (c) Alternating series error bound for ln(2)
print('(c) Alternating series error bound:')
print('   ln(2) = 1 - 1/2 + 1/3 - 1/4 + ...\n')
true_ln2 = np.log(2.0)
partial = 0.0
for k in range(1, 101):
    sign = 1.0 if k % 2 == 1 else -1.0
    partial += sign / k
    
    actual_error = abs(partial - true_ln2)
    bound = 1.0 / (k + 1)  # |a_{n+1}|
    
    if k in [5, 10, 20, 50, 100]:
        print(f'   n={k:3d}: actual error = {actual_error:.6e}, bound = {bound:.6e}, '
              f'bound holds: {actual_error <= bound + 1e-15}')

print(f'   The bound |a_{{n+1}}| always works for alternating decreasing series ✓\n')

# (d) Euler transform for series acceleration
print('(d) Euler transform acceleration:')
print('   Accelerating convergence of alternating harmonic series\n')

# Standard partial sums
partial_standard = 0.0
n_standard = -1
for k in range(1, 100001):
    sign = 1.0 if k % 2 == 1 else -1.0
    partial_standard += sign / k
    if abs(partial_standard - true_ln2) < 1e-8 and n_standard == -1:
        n_standard = k

# Euler transform: transforms a_0 - a_1 + a_2 - ... into
# ∑_{k=0}^{n} (-1)^k Δ^k(a_0) / 2^{k+1}
# where Δ^k are forward differences
N_euler = 30
a = np.zeros(N_euler + 1)
for k in range(N_euler + 1):
    a[k] = 1.0 / (k + 1)  # a_k = 1/(k+1)

# Compute forward differences Δ^k(a_0)
# Δ^0 a_0 = a_0
# Δ^k a_0 = Σ_{j=0}^{k} C(k,j)(-1)^{k-j} a_j
euler_sum = 0.0
n_euler = -1
for k in range(N_euler + 1):
    # Compute Δ^k a_0 using binomial coefficients
    delta_k = 0.0
    binom = 1.0  # C(k, 0) = 1
    for j in range(k + 1):
        sign = 1.0 if (k - j) % 2 == 0 else -1.0
        delta_k += sign * binom * a[j]
        if j < k:
            binom *= (k - j) / (j + 1.0)  # C(k, j+1) = C(k,j) × (k-j)/(j+1)
    
    euler_sum += delta_k / (2.0 ** (k + 1))
    if abs(euler_sum - true_ln2) < 1e-8 and n_euler == -1:
        n_euler = k

print(f'   Standard series: {n_standard if n_standard > 0 else ">100000"} terms for 8-digit accuracy')
print(f'   Euler transform: {n_euler} terms for 8-digit accuracy')
print(f'   Euler result:    {euler_sum:.10f}')
print(f'   True ln(2):      {true_ln2:.10f}')
print(f'   Speedup: ~{n_standard // max(n_euler, 1)}x fewer terms!')

## Exercise 16: Entropy & Information Theory Sums ⭐⭐⭐

**Topics**: Shannon entropy, cross-entropy, KL divergence, perplexity — all as explicit summations

### Tasks

**(a)** Implement **Shannon entropy** from scratch:
$$H(p) = -\sum_{i=1}^{n} p_i \log_2 p_i$$
- Handle $p_i = 0$ case (convention: $0 \log 0 = 0$)
- Compute for uniform distribution over $n$ classes and verify $H = \log_2 n$
- Compute for a peaked distribution $[0.9, 0.05, 0.03, 0.02]$ and explain why it's lower

**(b)** Implement **cross-entropy** and **KL divergence** from scratch:
$$H(p, q) = -\sum_i p_i \log q_i, \qquad D_{KL}(p \| q) = \sum_i p_i \log\frac{p_i}{q_i}$$
Verify that $D_{KL}(p \| q) = H(p, q) - H(p) \geq 0$ (Gibbs' inequality).

**(c)** **Perplexity** of a language model:
$$\text{PPL} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log p(w_i)\right) = \left(\prod_{i=1}^{N} p(w_i)\right)^{-1/N}$$
Given token probabilities, compute perplexity both ways (sum of logs vs product). Show why the log version is numerically better.

**(d)** **Entropy of softmax temperature**: Given logits $z$, compute the entropy of $\text{softmax}(z/T)$ for temperatures $T \in \{0.1, 0.5, 1.0, 2.0, 5.0\}$. Show that higher temperature → higher entropy (more uniform).

In [ ]:
# Exercise 16: Your Solution

import numpy as np

# (a) Shannon entropy
def shannon_entropy(p):
    """H(p) = -∑ p_i log2(p_i), with 0·log(0) = 0"""
    h = 0.0
    # YOUR CODE HERE
    return h

# Test: uniform distribution
# YOUR CODE HERE

# Test: peaked distribution
# YOUR CODE HERE

# (b) Cross-entropy and KL divergence
def cross_entropy(p, q):
    """H(p,q) = -∑ p_i log(q_i)"""
    # YOUR CODE HERE (use natural log)
    pass

def kl_divergence(p, q):
    """D_KL(p||q) = ∑ p_i log(p_i/q_i)"""
    # YOUR CODE HERE
    pass

# Verify: D_KL = H(p,q) - H(p)
# YOUR CODE HERE

# (c) Perplexity computation
token_probs = [0.1, 0.3, 0.05, 0.2, 0.15, 0.8, 0.4, 0.02, 0.5, 0.1]
# Log-space version: exp(-1/N ∑ log p_i)
# YOUR CODE HERE

# Product version: (∏ p_i)^{-1/N}
# YOUR CODE HERE

# (d) Temperature scaling and entropy
logits = np.array([2.0, 1.0, 0.5, -1.0, -2.0])
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]
# YOUR CODE HERE

In [ ]:
# Exercise 16: Solution ✅

import numpy as np

# (a) Shannon entropy
def shannon_entropy_bits(p):
    """H(p) = -∑ p_i log2(p_i), convention: 0·log(0) = 0"""
    h = 0.0
    for pi in p:
        if pi > 0:
            h -= pi * np.log2(pi)
    return h

# Uniform distribution
n_classes = 8
uniform = [1.0 / n_classes] * n_classes
h_uniform = shannon_entropy_bits(uniform)
print(f'(a) Shannon entropy:')
print(f'    Uniform({n_classes}): H = {h_uniform:.4f} bits')
print(f'    Expected log2({n_classes}) = {np.log2(n_classes):.4f} ✓')
assert abs(h_uniform - np.log2(n_classes)) < 1e-10

# Peaked distribution
peaked = [0.9, 0.05, 0.03, 0.02]
h_peaked = shannon_entropy_bits(peaked)
print(f'    Peaked {peaked}: H = {h_peaked:.4f} bits')
print(f'    Uniform({len(peaked)}): H = {np.log2(len(peaked)):.4f} bits')
print(f'    Peaked is lower — less uncertainty when one outcome dominates')

# Edge case with zero
with_zero = [0.5, 0.5, 0.0]
h_zero = shannon_entropy_bits(with_zero)
print(f'    [0.5, 0.5, 0.0]: H = {h_zero:.4f} (0·log(0)=0 convention)\n')

# (b) Cross-entropy and KL divergence (natural log)
def shannon_entropy_nats(p):
    h = 0.0
    for pi in p:
        if pi > 0:
            h -= pi * np.log(pi)
    return h

def cross_entropy_nats(p, q):
    """H(p,q) = -∑ p_i ln(q_i)"""
    ce = 0.0
    for pi, qi in zip(p, q):
        if pi > 0:
            ce -= pi * np.log(qi)
    return ce

def kl_divergence_nats(p, q):
    """D_KL(p||q) = ∑ p_i ln(p_i/q_i)"""
    kl = 0.0
    for pi, qi in zip(p, q):
        if pi > 0:
            kl += pi * np.log(pi / qi)
    return kl

p = [0.4, 0.3, 0.2, 0.1]
q = [0.25, 0.25, 0.25, 0.25]

hp = shannon_entropy_nats(p)
hpq = cross_entropy_nats(p, q)
kl_pq = kl_divergence_nats(p, q)

print(f'(b) Cross-entropy and KL divergence:')
print(f'    p = {p}')
print(f'    q = {q} (uniform)')
print(f'    H(p) = {hp:.6f} nats')
print(f'    H(p,q) = {hpq:.6f} nats')
print(f'    D_KL(p||q) = {kl_pq:.6f} nats')
print(f'    Verify: H(p,q) - H(p) = {hpq - hp:.6f} = D_KL ✓')
assert abs((hpq - hp) - kl_pq) < 1e-10
print(f'    D_KL ≥ 0: {kl_pq >= 0} (Gibbs\' inequality) ✓\n')

# (c) Perplexity
token_probs = [0.1, 0.3, 0.05, 0.2, 0.15, 0.8, 0.4, 0.02, 0.5, 0.1]
N = len(token_probs)

# Log-space: PPL = exp(-1/N ∑ log p_i)
log_sum = 0.0
for p_i in token_probs:
    log_sum += np.log(p_i)
ppl_log = np.exp(-log_sum / N)

# Product-space: PPL = (∏ p_i)^{-1/N}
product = 1.0
for p_i in token_probs:
    product *= p_i
ppl_prod = product ** (-1.0 / N)

print(f'(c) Perplexity:')
print(f'    Token probs: {token_probs}')
print(f'    Log version:     PPL = {ppl_log:.4f}')
print(f'    Product version: PPL = {ppl_prod:.4f}')
print(f'    Match: {abs(ppl_log - ppl_prod) < 1e-10} ✓')
print(f'    Product = {product:.2e} — underflows for long sequences!')
print(f'    Log version avoids this by staying in log space')
print(f'    Interpretation: model is as confused as choosing among {ppl_log:.1f} words uniformly\n')

# (d) Temperature scaling
logits = np.array([2.0, 1.0, 0.5, -1.0, -2.0])
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]

print(f'(d) Entropy vs temperature:')
print(f'    logits = {logits}')
print(f'    Max entropy (uniform over 5) = {np.log2(5):.4f} bits\n')

for T in temperatures:
    # Softmax(z/T) with stability
    scaled = logits / T
    max_s = scaled[0]
    for val in scaled:
        if val > max_s:
            max_s = val
    
    exp_vals = np.zeros(len(scaled))
    exp_sum = 0.0
    for i in range(len(scaled)):
        exp_vals[i] = np.exp(scaled[i] - max_s)
        exp_sum += exp_vals[i]
    
    probs = np.zeros(len(scaled))
    for i in range(len(scaled)):
        probs[i] = exp_vals[i] / exp_sum
    
    # Entropy in bits
    h = 0.0
    for pi in probs:
        if pi > 1e-30:
            h -= pi * np.log2(pi)
    
    bar = '█' * int(h / np.log2(5) * 30)
    print(f'    T={T:3.1f}: H={h:.4f} bits  top_p={probs[0]:.4f}  {bar}')

print(f'\n    T→0: argmax (greedy, H→0)')
print(f'    T→∞: uniform (random, H→log2(n))')
print(f'    T=1: standard softmax')

## Exercise 17: Expectation & Variance as Summations ⭐⭐⭐

**Topics**: Discrete expectation, variance decomposition, law of total expectation, moment generating functions

### Tasks

**(a)** Implement **expectation** and **variance** from definitions using explicit loops:
$$E[X] = \sum_i x_i p_i, \qquad \text{Var}(X) = \sum_i (x_i - \mu)^2 p_i = E[X^2] - (E[X])^2$$
Verify both variance formulas give the same result.

**(b)** **Weighted average as expectation**: Given model ensemble predictions $\hat{y}_1, \ldots, \hat{y}_K$ with weights $w_1, \ldots, w_K$, compute the ensemble prediction $\bar{y} = \sum_k w_k \hat{y}_k$ (where $\sum w_k = 1$). Then compute the ensemble **disagreement** (variance): $\sum_k w_k (\hat{y}_k - \bar{y})^2$.

**(c)** **Bias-variance decomposition**: For an estimator $\hat{\theta}$ over multiple samples, compute
$$\text{MSE} = \text{Bias}^2 + \text{Variance}$$
where $\text{Bias} = E[\hat{\theta}] - \theta$, $\text{Var} = E[(\hat{\theta} - E[\hat{\theta}])^2]$.
Demonstrate with sample mean estimator on different sample sizes.

**(d)** **Moment generating function**: For a discrete distribution, compute $M_X(t) = E[e^{tX}] = \sum_i p_i e^{t x_i}$ via explicit loop. Verify that $M'_X(0) = E[X]$ and $M''_X(0) = E[X^2]$ using numerical differentiation.

In [ ]:
# Exercise 17: Your Solution

import numpy as np

# (a) E[X] and Var(X) from definitions
values = np.array([1, 2, 3, 4, 5, 6], dtype=float)  # die
probs = np.array([1/6]*6)

# E[X] = ∑ x_i p_i
mean = 0.0
# YOUR CODE HERE

# Var(X) (definition form)
var_def = 0.0
# YOUR CODE HERE

# Var(X) (computational form: E[X²] - (E[X])²)
var_comp = 0.0
# YOUR CODE HERE

# (b) Model ensemble weighted average
K = 5
np.random.seed(7)
predictions = np.random.randn(K)
weights = np.array([0.35, 0.25, 0.20, 0.12, 0.08])

# Ensemble prediction: ȳ = ∑ w_k ŷ_k
# Disagreement: ∑ w_k (ŷ_k - ȳ)²
# YOUR CODE HERE

# (c) Bias-variance decomposition
# YOUR CODE HERE

# (d) Moment generating function
# YOUR CODE HERE

In [ ]:
# Exercise 17: Solution ✅

import numpy as np

# (a) Expectation and Variance from definitions
values = np.array([1, 2, 3, 4, 5, 6], dtype=float)
probs = np.array([1/6]*6)

# E[X] = ∑ x_i p_i
mean = 0.0
for i in range(len(values)):
    mean += values[i] * probs[i]

# Var(X) = ∑ (x_i - μ)² p_i  (definition form)
var_def = 0.0
for i in range(len(values)):
    var_def += (values[i] - mean) ** 2 * probs[i]

# Var(X) = E[X²] - (E[X])²  (computational form)
ex2 = 0.0
for i in range(len(values)):
    ex2 += values[i] ** 2 * probs[i]
var_comp = ex2 - mean ** 2

print(f'(a) Fair die:')
print(f'    E[X] = {mean:.4f} (expected: 3.5)')
print(f'    Var(X) definition:    {var_def:.4f}')
print(f'    Var(X) computational: {var_comp:.4f}')
assert abs(var_def - var_comp) < 1e-10
print(f'    Both formulas match ✓')
print(f'    Expected: 35/12 = {35/12:.4f} ✓\n')

# (b) Model ensemble
K = 5
np.random.seed(7)
predictions = np.random.randn(K)
weights = np.array([0.35, 0.25, 0.20, 0.12, 0.08])

# Normalize weights (should sum to 1)
w_sum = 0.0
for w in weights:
    w_sum += w
for i in range(len(weights)):
    weights[i] /= w_sum

# Ensemble mean: ȳ = ∑ w_k ŷ_k
y_bar = 0.0
for k in range(K):
    y_bar += weights[k] * predictions[k]

# Disagreement: ∑ w_k (ŷ_k - ȳ)²
disagreement = 0.0
for k in range(K):
    disagreement += weights[k] * (predictions[k] - y_bar) ** 2

print(f'(b) Model ensemble:')
print(f'    Predictions: {predictions.round(4)}')
print(f'    Weights:     {weights.round(4)}')
print(f'    Ensemble ȳ:  {y_bar:.4f}')
print(f'    Disagreement: {disagreement:.4f}')
print(f'    High disagreement → models uncertain → useful for active learning\n')

# (c) Bias-variance decomposition
print(f'(c) Bias-variance decomposition:')
print(f'    Estimator: sample mean of N draws from N(μ=5, σ²=4)')
true_theta = 5.0
true_sigma = 2.0
np.random.seed(42)

for N in [5, 20, 100, 500]:
    n_experiments = 2000
    estimates = np.zeros(n_experiments)
    
    for exp in range(n_experiments):
        # Draw N samples, compute sample mean
        total = 0.0
        for _ in range(N):
            total += np.random.normal(true_theta, true_sigma)
        estimates[exp] = total / N
    
    # Bias = E[θ̂] - θ
    mean_estimate = 0.0
    for est in estimates:
        mean_estimate += est
    mean_estimate /= n_experiments
    bias = mean_estimate - true_theta
    
    # Variance = E[(θ̂ - E[θ̂])²]
    variance = 0.0
    for est in estimates:
        variance += (est - mean_estimate) ** 2
    variance /= n_experiments
    
    # MSE = E[(θ̂ - θ)²]
    mse = 0.0
    for est in estimates:
        mse += (est - true_theta) ** 2
    mse /= n_experiments
    
    print(f'    N={N:3d}: Bias²={bias**2:.6f}, Var={variance:.6f}, '
          f'Sum={bias**2 + variance:.6f}, MSE={mse:.6f}')

print(f'    MSE ≈ Bias² + Var ✓  (sample mean is unbiased: Bias≈0)\n')

# (d) Moment generating function
print(f'(d) Moment generating function:')
# Bernoulli(p=0.3): X ∈ {0, 1}
x_vals = np.array([0.0, 1.0])
p_vals = np.array([0.7, 0.3])

# M_X(t) = ∑ p_i e^{t x_i}
def mgf(t, x_vals, p_vals):
    result = 0.0
    for i in range(len(x_vals)):
        result += p_vals[i] * np.exp(t * x_vals[i])
    return result

# E[X] via definition
ex_direct = 0.0
for i in range(len(x_vals)):
    ex_direct += x_vals[i] * p_vals[i]

# E[X²] via definition
ex2_direct = 0.0
for i in range(len(x_vals)):
    ex2_direct += x_vals[i] ** 2 * p_vals[i]

# Numerical derivatives of MGF at t=0
h = 1e-5
mgf_prime_0 = (mgf(h, x_vals, p_vals) - mgf(-h, x_vals, p_vals)) / (2 * h)
mgf_double_0 = (mgf(h, x_vals, p_vals) - 2 * mgf(0, x_vals, p_vals) + mgf(-h, x_vals, p_vals)) / (h ** 2)

print(f'    Bernoulli(0.3):')
print(f'    E[X] direct:      {ex_direct:.6f}')
print(f'    M\'(0) numerical:  {mgf_prime_0:.6f}  ✓')
print(f'    E[X²] direct:     {ex2_direct:.6f}')
print(f'    M\'\'(0) numerical: {mgf_double_0:.6f}  ✓')
print(f'    MGFs encode ALL moments of a distribution!')

## Exercise 18: BM25 Scoring & TF-IDF ⭐⭐⭐

**Topics**: Summation in information retrieval, term frequency, inverse document frequency

### Tasks

**(a)** Implement **TF-IDF** from scratch for a small corpus:
$$\text{tf-idf}(t, d) = \text{tf}(t, d) \cdot \log\frac{N}{|\{d' : t \in d'\}|}$$
where $\text{tf}(t, d)$ is the count of term $t$ in document $d$, and $N$ is the total number of documents.

**(b)** Implement the **BM25 scoring function**:
$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$
where $\text{IDF}(t) = \log\frac{N - n(t) + 0.5}{n(t) + 0.5}$, $k_1 = 1.5$, $b = 0.75$.

Use only explicit loops — no sklearn or library functions.

**(c)** Rank a set of 5 documents against a query using your BM25 implementation.

**(d)** Show how **document length normalization** ($b$ parameter) affects ranking. Compare $b = 0$ (no normalization) vs $b = 1$ (full normalization) for documents of varying length.

In [ ]:
# Exercise 18: Your Solution

import numpy as np

# Corpus
docs = [
    "the cat sat on the mat",
    "the dog chased the cat",
    "a cat and a dog are friends",
    "the mat is on the floor",
    "dogs and cats are great pets"
]
query = "cat dog"

# (a) TF-IDF from scratch
# Step 1: Tokenize
# Step 2: Compute tf(t, d) for each term in each doc
# Step 3: Compute idf(t) = log(N / df(t))
# Step 4: tf_idf(t, d) = tf * idf
# YOUR CODE HERE

# (b) BM25 scoring
# YOUR CODE HERE: k1=1.5, b=0.75

# (c) Rank documents
# YOUR CODE HERE

# (d) Effect of b parameter
# YOUR CODE HERE

In [ ]:
# Exercise 18: Solution ✅

import numpy as np

docs = [
    "the cat sat on the mat",
    "the dog chased the cat",
    "a cat and a dog are friends",
    "the mat is on the floor",
    "dogs and cats are great pets"
]
query = "cat dog"

# Tokenize all docs
doc_tokens = []
for doc in docs:
    tokens = doc.lower().split()
    doc_tokens.append(tokens)
query_tokens = query.lower().split()
N = len(docs)

# (a) TF-IDF from scratch
print('(a) TF-IDF:')

# Build vocabulary
vocab = set()
for tokens in doc_tokens:
    for t in tokens:
        vocab.add(t)
vocab = sorted(vocab)

# Compute tf(t, d) — raw count
def compute_tf(term, tokens):
    count = 0
    for t in tokens:
        if t == term:
            count += 1
    return count

# Compute df(t) — number of docs containing term
def compute_df(term, doc_tokens):
    count = 0
    for tokens in doc_tokens:
        found = False
        for t in tokens:
            if t == term:
                found = True
                break
        if found:
            count += 1
    return count

# Show TF-IDF for query terms
for qt in query_tokens:
    df = compute_df(qt, doc_tokens)
    idf = np.log(N / max(df, 1))
    print(f'  term="{qt}": df={df}, idf=ln({N}/{df})={idf:.4f}')
    for d_idx, tokens in enumerate(doc_tokens):
        tf = compute_tf(qt, tokens)
        tfidf = tf * idf
        if tf > 0:
            print(f'    doc{d_idx}: tf={tf}, tf-idf={tfidf:.4f}')
print()

# (b) BM25 scoring
print('(b) BM25 scoring:')
k1 = 1.5
b = 0.75

# Compute average document length
total_len = 0
for tokens in doc_tokens:
    total_len += len(tokens)
avgdl = total_len / N

def bm25_score(query_tokens, doc_tokens_single, all_doc_tokens, k1, b, avgdl):
    """Compute BM25 score for a single document against a query."""
    score = 0.0
    dl = len(doc_tokens_single)
    N = len(all_doc_tokens)
    
    for qt in query_tokens:
        # Term frequency in this doc
        f_td = compute_tf(qt, doc_tokens_single)
        
        # Document frequency
        n_t = compute_df(qt, all_doc_tokens)
        
        # IDF component (BM25 variant)
        idf = np.log((N - n_t + 0.5) / (n_t + 0.5))
        
        # TF component with length normalization
        tf_norm = (f_td * (k1 + 1)) / (f_td + k1 * (1 - b + b * dl / avgdl))
        
        score += idf * tf_norm
    
    return score

# (c) Rank documents
print(f'  Query: "{query}"')
print(f'  k1={k1}, b={b}, avgdl={avgdl:.1f}\n')

scores = []
for d_idx in range(N):
    s = bm25_score(query_tokens, doc_tokens[d_idx], doc_tokens, k1, b, avgdl)
    scores.append((d_idx, s))

# Sort by score descending (manual bubble sort for raw implementation)
for i in range(len(scores)):
    for j in range(i + 1, len(scores)):
        if scores[j][1] > scores[i][1]:
            scores[i], scores[j] = scores[j], scores[i]

print('  Ranking:')
for rank, (d_idx, s) in enumerate(scores):
    print(f'    #{rank+1}: doc{d_idx} (score={s:.4f}) "{docs[d_idx]}"')

# (d) Effect of b parameter
print(f'\n(d) Effect of length normalization (b):')
for b_val in [0.0, 0.25, 0.5, 0.75, 1.0]:
    print(f'\n  b={b_val}:')
    b_scores = []
    for d_idx in range(N):
        s = bm25_score(query_tokens, doc_tokens[d_idx], doc_tokens, k1, b_val, avgdl)
        b_scores.append((d_idx, s))
    
    # Sort
    for i in range(len(b_scores)):
        for j in range(i + 1, len(b_scores)):
            if b_scores[j][1] > b_scores[i][1]:
                b_scores[i], b_scores[j] = b_scores[j], b_scores[i]
    
    for rank, (d_idx, s) in enumerate(b_scores[:3]):
        dl = len(doc_tokens[d_idx])
        print(f'    #{rank+1}: doc{d_idx} (score={s:.4f}, len={dl})')

print(f'\n  b=0: no length penalty — long docs with many matches win')
print(f'  b=1: full normalization — penalizes longer documents')
print(f'  b=0.75: standard BM25 default — balanced')

## Exercise 19: Gradient Accumulation & Mixed Precision ⭐⭐⭐

**Topics**: Gradient accumulation as summation, effective batch size, mixed-precision loss scaling

### Tasks

**(a)** **Gradient accumulation**: When GPU memory can't fit a large batch, we accumulate gradients over $K$ micro-batches:
$$g_{\text{effective}} = \frac{1}{K}\sum_{k=1}^{K} g_k$$
Simulate this: generate random "gradients" for each micro-batch and compute the accumulated gradient. Verify it equals the gradient of the full batch.

**(b)** **Loss scaling for mixed precision**: In float16 training, small gradients underflow. Loss scaling multiplies the loss by a factor $S$, computes gradients (which are scaled by $S$), then divides:
$$g_{\text{true}} = \frac{1}{S}\sum_k \nabla L_k \cdot S$$
Implement this pipeline and show that without scaling, small gradients vanish in float16.

**(c)** **Running gradient statistics**: Implement exponential moving average of gradient moments (as used in Adam):
$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \qquad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
Show how bias correction affects early steps.

**(d)** **Gradient clipping by global norm**: Given gradients for multiple parameter tensors, compute the global norm $\|g\| = \sqrt{\sum_l \|g_l\|^2}$ and clip: if $\|g\| > \tau$, scale all gradients by $\tau / \|g\|$.

In [ ]:
# Exercise 19: Your Solution

import numpy as np

# (a) Gradient accumulation
K = 4  # micro-batches
micro_batch_size = 8
total_batch_size = K * micro_batch_size
param_dim = 10

np.random.seed(19)
# YOUR CODE HERE: simulate gradient accumulation

# (b) Loss scaling for mixed precision
# YOUR CODE HERE: show float16 underflow and fix with scaling

# (c) Adam-style running moments
# YOUR CODE HERE: implement EMA with bias correction

# (d) Gradient clipping by global norm
# YOUR CODE HERE

In [ ]:
# Exercise 19: Solution ✅

import numpy as np

# (a) Gradient accumulation
K = 4  # micro-batches
micro_batch_size = 8
total_batch_size = K * micro_batch_size
param_dim = 10

np.random.seed(19)

# Generate all data at once (full batch)
all_data = np.random.randn(total_batch_size, param_dim)

# Full-batch gradient: mean of all samples
full_grad = np.zeros(param_dim)
for i in range(total_batch_size):
    for j in range(param_dim):
        full_grad[j] += all_data[i, j]
for j in range(param_dim):
    full_grad[j] /= total_batch_size

# Accumulated gradient: average of K micro-batch gradients
accum_grad = np.zeros(param_dim)
for k in range(K):
    micro_grad = np.zeros(param_dim)
    start = k * micro_batch_size
    for i in range(start, start + micro_batch_size):
        for j in range(param_dim):
            micro_grad[j] += all_data[i, j]
    for j in range(param_dim):
        micro_grad[j] /= micro_batch_size
    # Accumulate: add micro_grad / K
    for j in range(param_dim):
        accum_grad[j] += micro_grad[j] / K

print(f'(a) Gradient accumulation:')
print(f'    Full batch ({total_batch_size} samples): {full_grad[:5].round(4)}...')
print(f'    Accumulated ({K}×{micro_batch_size}):     {accum_grad[:5].round(4)}...')
max_diff = 0.0
for j in range(param_dim):
    diff = abs(full_grad[j] - accum_grad[j])
    if diff > max_diff:
        max_diff = diff
print(f'    Max difference: {max_diff:.2e} ✓')
print(f'    Effective batch size = K × micro_batch = {total_batch_size}\n')

# (b) Loss scaling for mixed precision
print(f'(b) Loss scaling for mixed precision:')
# Simulate small gradients that underflow in float16
grads_f32 = np.array([1e-6, 3e-7, 8e-8, 2e-5, 5e-9], dtype=np.float32)

# Convert to float16 without scaling
grads_f16_raw = grads_f32.astype(np.float16)
print(f'    Original f32 grads: {grads_f32}')
print(f'    Direct f16 (no scaling): {grads_f16_raw}')
print(f'    Zeros (underflow): {np.sum(grads_f16_raw == 0)} of {len(grads_f16_raw)}')

# With loss scaling: multiply by S before f16, divide after
S = np.float32(65536.0)  # common scaling factor
grads_scaled = np.zeros(len(grads_f32), dtype=np.float16)
grads_recovered = np.zeros(len(grads_f32), dtype=np.float32)

for i in range(len(grads_f32)):
    scaled = grads_f32[i] * S
    grads_scaled[i] = np.float16(scaled)  # now representable in f16
    grads_recovered[i] = np.float32(grads_scaled[i]) / S

print(f'    Scaled f16 (×{S:.0f}):     {grads_scaled}')
print(f'    Recovered f32 (÷{S:.0f}): {grads_recovered}')
print(f'    Zeros after scaling: {np.sum(grads_recovered == 0)}')
print(f'    Loss scaling saves training from gradient underflow!\n')

# (c) Adam-style running moments (EMA with bias correction)
print(f'(c) Adam running moments:')
beta1, beta2 = 0.9, 0.999
np.random.seed(42)

# Simulate 20 gradient steps
m = 0.0  # first moment (mean)
v = 0.0  # second moment (uncentered variance)

print(f'    β1={beta1}, β2={beta2}')
print(f'    {"step":>4s} {"g_t":>8s} {"m_t":>8s} {"m_hat":>8s} {"v_t":>10s} {"v_hat":>10s}')
print(f'    {"-"*48}')

for t in range(1, 21):
    g_t = np.random.randn() + 2.0  # true gradient ≈ 2.0
    
    # Update moments
    m = beta1 * m + (1 - beta1) * g_t
    v = beta2 * v + (1 - beta2) * g_t ** 2
    
    # Bias correction
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    
    if t <= 5 or t in [10, 20]:
        print(f'    {t:4d} {g_t:8.4f} {m:8.4f} {m_hat:8.4f} {v:10.4f} {v_hat:10.4f}')

print(f'\n    Without bias correction: m starts near 0 (severely biased early)')
print(f'    With correction: m̂ immediately estimates true mean ≈ 2.0')
print(f'    β2=0.999 means v adapts slowly — stable denominator in Adam\n')

# (d) Gradient clipping by global norm
print(f'(d) Gradient clipping by global norm:')
# Simulate gradients for 3 parameter tensors
np.random.seed(99)
param_grads = [
    np.random.randn(4, 3) * 5,   # layer 1: 4×3
    np.random.randn(3, 2) * 10,  # layer 2: 3×2 (larger grads)
    np.random.randn(2) * 3,      # bias: size 2
]

# Compute global norm: sqrt(∑_l ||g_l||²)
global_norm_sq = 0.0
for g in param_grads:
    for val in g.flat:
        global_norm_sq += val ** 2
global_norm = np.sqrt(global_norm_sq)

tau = 5.0  # clipping threshold
print(f'    Global norm: {global_norm:.4f}')
print(f'    Threshold τ: {tau}')

if global_norm > tau:
    scale = tau / global_norm
    clipped_grads = []
    for g in param_grads:
        clipped = np.zeros_like(g)
        for idx in np.ndindex(g.shape):
            clipped[idx] = g[idx] * scale
        clipped_grads.append(clipped)
    
    # Verify clipped norm
    clipped_norm_sq = 0.0
    for g in clipped_grads:
        for val in g.flat:
            clipped_norm_sq += val ** 2
    clipped_norm = np.sqrt(clipped_norm_sq)
    
    print(f'    Scaling: {scale:.4f}')
    print(f'    Clipped norm: {clipped_norm:.4f} ≈ τ ✓')
    print(f'    Direction preserved, magnitude bounded')
else:
    print(f'    No clipping needed (norm ≤ τ)')

print(f'    Used in transformer training to prevent gradient explosions')

## Exercise 20: Full Transformer Forward Pass ⭐⭐⭐

**Topics**: Combining all summation patterns into a complete transformer block

### Tasks

Build a **single transformer decoder block** from scratch using only explicit loops and basic numpy operations. This exercise ties together everything from the notebook.

**(a)** **Layer normalization** using summation:
$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}, \quad \mu = \frac{1}{d}\sum_{j=1}^d x_j, \quad \sigma^2 = \frac{1}{d}\sum_{j=1}^d (x_j - \mu)^2$$

**(b)** **Causal self-attention** combining:
- Linear projections (matrix multiply as summation)
- Scaled dot-product attention with causal mask
- Softmax with stability trick

**(c)** **Feed-forward network**: Two linear layers with ReLU:
$$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

**(d)** **Full block**: Combine with residual connections:
$$h = x + \text{Attention}(\text{LN}(x))$$
$$\text{out} = h + \text{FFN}(\text{LN}(h))$$

Count the total FLOPs for your implementation.

In [ ]:
# Exercise 20: Your Solution

import numpy as np

# Dimensions
seq_len = 4   # sequence length
d_model = 8   # model dimension
d_ff = 16     # feed-forward hidden dimension
n_heads = 2   # number of attention heads
d_head = d_model // n_heads  # head dimension

np.random.seed(20)
x = np.random.randn(seq_len, d_model)  # input: (seq_len, d_model)

# (a) Layer normalization
def layer_norm(x, eps=1e-5):
    """Normalize each row (token) independently."""
    out = np.zeros_like(x)
    # YOUR CODE HERE: compute mean, var, normalize for each row
    return out

# (b) Causal self-attention
def causal_self_attention(x, Wq, Wk, Wv, Wo, n_heads):
    """Multi-head causal self-attention using only loops."""
    # YOUR CODE HERE
    pass

# (c) Feed-forward network
def feed_forward(x, W1, b1, W2, b2):
    """FFN(x) = max(0, xW1 + b1)W2 + b2"""
    # YOUR CODE HERE
    pass

# (d) Full transformer block
# YOUR CODE HERE

In [ ]:
# Exercise 20: Solution ✅

import numpy as np

# Dimensions
seq_len = 4
d_model = 8
d_ff = 16
n_heads = 2
d_head = d_model // n_heads

np.random.seed(20)
x = np.random.randn(seq_len, d_model)
flops_total = 0

# ============================================================
# (a) Layer Normalization
# ============================================================
def layer_norm_raw(x, eps=1e-5):
    """LayerNorm using explicit loops on each token."""
    rows, cols = x.shape
    out = np.zeros_like(x)
    flops = 0
    
    for i in range(rows):
        # Mean: μ = (1/d) Σ_j x[i,j]
        mu = 0.0
        for j in range(cols):
            mu += x[i, j]
        mu /= cols
        flops += cols + 1  # cols adds + 1 divide
        
        # Variance: σ² = (1/d) Σ_j (x[i,j] - μ)²
        var = 0.0
        for j in range(cols):
            diff = x[i, j] - mu
            var += diff * diff
        var /= cols
        flops += 3 * cols + 1  # sub, mul, add per j + divide
        
        # Normalize: x̂[i,j] = (x[i,j] - μ) / sqrt(σ² + ε)
        inv_std = 1.0 / np.sqrt(var + eps)
        for j in range(cols):
            out[i, j] = (x[i, j] - mu) * inv_std
        flops += 2 * cols + 2  # sub+mul per j, sqrt+div
    
    return out, flops

x_norm, ln_flops = layer_norm_raw(x)
flops_total += ln_flops
print(f'(a) Layer Normalization:')
print(f'    Input shape: {x.shape}')
print(f'    Output mean per token: {[f"{m:.4f}" for m in x_norm.mean(axis=1)]}')
print(f'    Output var per token:  {[f"{v:.4f}" for v in x_norm.var(axis=1)]}')
print(f'    FLOPs: {ln_flops}\n')

# ============================================================
# (b) Causal Self-Attention (multi-head)
# ============================================================
# Initialize projection matrices
Wq = np.random.randn(d_model, d_model) * 0.1
Wk = np.random.randn(d_model, d_model) * 0.1
Wv = np.random.randn(d_model, d_model) * 0.1
Wo = np.random.randn(d_model, d_model) * 0.1

def matmul_raw(A, B):
    """Matrix multiply using triple loop. Returns (result, flops)."""
    M, K = A.shape
    K2, N = B.shape
    assert K == K2
    C = np.zeros((M, N))
    for i in range(M):
        for j in range(N):
            for k in range(K):
                C[i, j] += A[i, k] * B[k, j]
    return C, 2 * M * N * K

def softmax_stable_row(row):
    """Stable softmax for a 1D array."""
    max_val = row[0]
    for v in row:
        if v > max_val:
            max_val = v
    exp_vals = np.zeros(len(row))
    exp_sum = 0.0
    for i in range(len(row)):
        exp_vals[i] = np.exp(row[i] - max_val)
        exp_sum += exp_vals[i]
    for i in range(len(row)):
        exp_vals[i] /= exp_sum
    return exp_vals

# Q, K, V projections
Q, q_flops = matmul_raw(x_norm, Wq)  # (seq, d_model)
K_proj, k_flops = matmul_raw(x_norm, Wk)
V, v_flops = matmul_raw(x_norm, Wv)
attn_flops = q_flops + k_flops + v_flops

# Reshape for multi-head: (seq, d_model) → process as (n_heads, seq, d_head)
# Manual reshape via indexing
attn_output = np.zeros((seq_len, d_model))

for h in range(n_heads):
    d_start = h * d_head
    d_end = (h + 1) * d_head
    
    # Extract head slices: (seq_len, d_head)
    Q_h = Q[:, d_start:d_end]
    K_h = K_proj[:, d_start:d_end]
    V_h = V[:, d_start:d_end]
    
    # Scores: Q_h @ K_h^T / sqrt(d_head)
    scores = np.zeros((seq_len, seq_len))
    scale = 1.0 / np.sqrt(d_head)
    for i in range(seq_len):
        for j in range(seq_len):
            dot = 0.0
            for k in range(d_head):
                dot += Q_h[i, k] * K_h[j, k]
            scores[i, j] = dot * scale
    attn_flops += 2 * seq_len * seq_len * d_head + seq_len * seq_len
    
    # Causal mask: set future positions to -inf
    for i in range(seq_len):
        for j in range(seq_len):
            if j > i:
                scores[i, j] = -1e9
    
    # Softmax per row
    attn_weights = np.zeros((seq_len, seq_len))
    for i in range(seq_len):
        attn_weights[i] = softmax_stable_row(scores[i])
    attn_flops += seq_len * seq_len * 3  # exp, sum, div per element
    
    # Weighted sum of values
    for i in range(seq_len):
        for d in range(d_head):
            weighted = 0.0
            for j in range(seq_len):
                weighted += attn_weights[i, j] * V_h[j, d]
            attn_output[i, d_start + d] = weighted
    attn_flops += 2 * seq_len * seq_len * d_head

# Output projection
attn_projected, wo_flops = matmul_raw(attn_output, Wo)
attn_flops += wo_flops
flops_total += attn_flops

print(f'(b) Causal Self-Attention ({n_heads} heads):')
print(f'    Q, K, V shape: ({seq_len}, {d_model})')
print(f'    Attention weights [0] (causal):')
print(f'    {attn_weights[0].round(3)}  ← can only attend to past')
print(f'    Output shape: {attn_projected.shape}')
print(f'    FLOPs: {attn_flops}\n')

# ============================================================
# (c) Feed-Forward Network
# ============================================================
W1 = np.random.randn(d_model, d_ff) * 0.1
b1 = np.zeros(d_ff)
W2 = np.random.randn(d_ff, d_model) * 0.1
b2 = np.zeros(d_model)

# FFN input: layer_norm of residual
h = np.zeros_like(x)
for i in range(seq_len):
    for j in range(d_model):
        h[i, j] = x[i, j] + attn_projected[i, j]  # residual connection

h_norm, ln2_flops = layer_norm_raw(h)

# Linear 1 + ReLU
hidden, ff1_flops = matmul_raw(h_norm, W1)
ff_flops = ff1_flops
for i in range(seq_len):
    for j in range(d_ff):
        hidden[i, j] = hidden[i, j] + b1[j]  # bias
        if hidden[i, j] < 0:                   # ReLU
            hidden[i, j] = 0.0
ff_flops += seq_len * d_ff * 2  # bias add + compare

# Linear 2
ffn_out, ff2_flops = matmul_raw(hidden, W2)
ff_flops += ff2_flops
for i in range(seq_len):
    for j in range(d_model):
        ffn_out[i, j] = ffn_out[i, j] + b2[j]
ff_flops += seq_len * d_model
flops_total += ff_flops + ln2_flops

print(f'(c) Feed-Forward Network:')
print(f'    Hidden dim: {d_ff} (4× model dim is standard)')
print(f'    ReLU sparsity: {np.sum(hidden == 0) / hidden.size:.1%} zeros')
print(f'    Output shape: {ffn_out.shape}')
print(f'    FLOPs: {ff_flops}\n')

# ============================================================
# (d) Full Block = Residual + FFN output
# ============================================================
block_out = np.zeros_like(x)
for i in range(seq_len):
    for j in range(d_model):
        block_out[i, j] = h[i, j] + ffn_out[i, j]  # second residual

print(f'(d) Full Transformer Block:')
print(f'    Input:  x ({seq_len}, {d_model})')
print(f'    Output: ({seq_len}, {d_model})')
print(f'    Block = x + Attn(LN(x)) → h')
print(f'           h + FFN(LN(h))   → out')
print(f'')
print(f'    === FLOPs Breakdown ===')
print(f'    LayerNorm ×2: {ln_flops + ln2_flops:,}')
print(f'    Attention:    {attn_flops:,}')
print(f'    FFN:          {ff_flops:,}')
print(f'    Total:        {flops_total:,}')
print(f'')
print(f'    Every summation pattern we learned appears here:')
print(f'    ✓ Basic Σ: means, variances (LayerNorm)')
print(f'    ✓ Matrix multiply as Σ_k: all linear projections')
print(f'    ✓ Softmax (exp + normalize): attention weights')
print(f'    ✓ Weighted sum: attention output')
print(f'    ✓ Residual connection: skip connection addition')
print(f'    ✓ Causal mask: restricting summation range')
print(f'    ✓ Multi-head: independent parallel summations')

---

# Summary & Mastery Checklist

## What You've Built

| # | Exercise | Key Concept | ML Connection |
|---|----------|-------------|---------------|
| 1 | Expand Sum/Product | Summation identity elements | Accumulator pattern |
| 2 | Write Sigma/Pi | Closed-form from pattern | Recognizing series |
| 3 | Linearity & Telescoping | Algebraic manipulation | ResNet residuals |
| 4 | Index Shifting & Interchange | Bound transformation | Gauss's trick |
| 5 | Geometric Series | Finite/infinite sums, EMA | Exponential moving average |
| 6 | Product-to-Sum via Log | Log-space computation | NLL, MLE |
| 7 | Softmax | Stable implementation, Jacobian | Every classifier |
| 8 | Cross-Entropy | Loss decomposition, label smoothing | Training objective |
| 9 | Attention | Weighted sum with masking | Transformer core |
| 10 | Matrix Multiply | Triple-loop, FLOPs analysis | Every layer |
| 11 | Einstein Summation | einsum notation | Efficient tensor ops |
| 12 | Double Sums | Order interchange, convolution | CNNs, dependent bounds |
| 13 | Kahan Summation | Numerical stability | fp16 training |
| 14 | Infinite Products | Convergence, scaling | Weight initialization |
| 15 | Series Convergence | Taylor series, acceleration | Approximation theory |
| 16 | Entropy & Info Theory | H, KL, perplexity | LLM evaluation |
| 17 | Expectation & Variance | Moments, bias-variance | Model evaluation |
| 18 | BM25 & TF-IDF | Scoring as summation | Information retrieval |
| 19 | Gradient Accumulation | Effective batch, Adam, clipping | Training infrastructure |
| 20 | Full Transformer Block | All patterns combined | Modern LLMs |

## Mastery Levels

- **⭐ Essential (1-6)**: Can you fluently read and write sigma/pi notation?  
- **⭐⭐ Applied (7-14)**: Can you implement ML primitives from scratch?  
- **⭐⭐⭐ Production (15-20)**: Can you reason about numerical stability, FLOPs, and system design?

> **Every modern AI model is built from the summation patterns in these exercises.**